# 01 — Preparação e construção inicial da tabela mestra

**Escopo do notebook:** inspeção das fontes, preparação das identificações e construção de um calendário diário inicial com contagens de alarmes. O alvo, as sequências e o modelo serão desenvolvidos em etapas posteriores. Os arquivos de origem permanecem intactos.

A futura unidade de análise será **estação + plataforma + porta + dia**, verificando se o tipo de ativo também precisa compor a chave. O alvo pretendido é a ocorrência de falha entre D+1 e D+7. ZM01 é evidência principal; ZM02 e ZM03 descrevem histórico de manutenção. Manutenção programada ou preventiva não equivale automaticamente a falha.

O notebook utiliza exclusivamente os quatro arquivos tratados em `data/`. As abas de auditoria são inspecionadas separadamente, sem concatená-las às bases principais.

In [1]:
from pathlib import Path
import re
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)

# Funciona iniciando o Jupyter na raiz ou em uma subpasta do repositório.
candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos
             if (p / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("O notebook precisa ser executado a partir do repositório que contém data/.")

FONTES = {
    "Alarmes": (RAIZ / "data/Alarmes/alarmes_trata_limpos_consolidado.csv", None),
    "ZM01": (RAIZ / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx", "ZM01_limpa"),
    "ZM02": (RAIZ / "data/Corretiva Programada/processados/ZM02_limpa.xlsx", "ZM02_limpa"),
    "ZM03": (RAIZ / "data/Preventiva/preventiva_zm03_limpa.csv", None),
}
for nome, (arquivo, _) in FONTES.items():
    if not arquivo.is_file():
        raise FileNotFoundError(f"{nome}: {arquivo}")
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"Raiz: {RAIZ}")


Python: 3.13.14 | pandas: 2.3.1
Raiz: C:\Users\Inteli\Downloads\EC-M7\g04


## 1. Inventário e leitura

Os CSVs são lidos em UTF-8. As abas são escolhidas explicitamente para não confundir a base limpa com auditorias ou subconjuntos. A contagem de linhas abaixo corresponde aos registros lidos, não à quantidade de episódios de falha.

In [2]:
bases = {}
auditorias = {}
inventario = []
for nome, (arquivo, principal) in FONTES.items():
    if principal is None:
        df = pd.read_csv(arquivo, encoding="utf-8-sig", low_memory=False)
        bases[nome] = df
        inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                           "aba": "(CSV)", "papel": "principal",
                           "linhas": len(df), "colunas": len(df.columns)})
    else:
        with pd.ExcelFile(arquivo) as livro:
            for aba in livro.sheet_names:
                df = pd.read_excel(livro, sheet_name=aba)
                if aba == principal:
                    bases[nome] = df
                else:
                    auditorias[(nome, aba)] = df
                inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                                   "aba": aba, "papel": "principal" if aba == principal else "auditoria",
                                   "linhas": len(df), "colunas": len(df.columns)})
display(pd.DataFrame(inventario))
for (nome, aba), df in auditorias.items():
    if aba == "Decisoes_limpeza":
        print(f"Decisões de limpeza — {nome}")
        display(df)


,fonte,arquivo,aba,papel,linhas,colunas
0,Alarmes,data\Alarmes\alarmes_trata_limpos_consolidado.csv,(CSV),principal,742506,15
1,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,ZM01_limpa,principal,10543,27
2,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Outliers_removidos,auditoria,103,27
3,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
4,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Decisoes_limpeza,auditoria,7,2
5,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,ZM02_limpa,principal,1163,27
6,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_extremas,auditoria,12,27
7,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_zero,auditoria,5,27
8,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
9,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Decisoes_limpeza,auditoria,9,2


Decisões de limpeza — ZM01


,campo_ou_regra,decisao
0,iqm,Coluna removida por conter 100% de valores ausentes.
1,duplicatas,Removidas 1 linha(s) exatamente duplicada(s).
2,duracao_avaria_horas,"Removidos 103 registros acima do percentil 99 (25.00 horas), preservados separadamente para audi..."
3,fim_da_avaria,Ausências preservadas; não foi realizada imputação.
4,data_analise,Ausências preservadas; marca e data de análise não são equivalentes.
5,campos_catalogo,"Ausências de objeto, defeito, causa e ação foram preservadas."
6,ordens_ausentes,Registros sem ordem foram mantidos quando possuíam nota e dados úteis.


Decisões de limpeza — ZM02


,campo_ou_regra,decisao
0,falha,Removida por conter 100% de valores ausentes.
1,iqm,Removida por conter 100% de valores ausentes.
2,duplicatas,Nenhuma duplicata exata encontrada após a limpeza.
3,ordens_repetidas,Cinco ordens repetidas foram mantidas porque apresentam divergências em campos técnicos.
4,ordens_ausentes,"Cinco registros sem ordem foram mantidos por possuírem nota, descrição e localização."
5,valores_ausentes,As ausências foram preservadas sem imputação.
6,duracao_extrema,Foram sinalizados 12 registros acima do percentil 99 (6104.70 horas). Os registros foram mantido...
7,duracao_zero,"Foram sinalizados 5 registros com duração igual a zero, mantidos sem imputação."
8,colunas_esparsas,As colunas comentarios_classificacao_engenharia e analise_datetime foram removidas da base final...


## 2. Colunas, tipos e ausências

Os tipos são inferidos na leitura. Esta inspeção preserva os valores ausentes e todos os registros. As amostras exibem somente identificação e datas, evitando reproduzir textos longos das ordens.

In [3]:
AMOSTRAS = {
    "Alarmes": ["TAG", "estacao_extraida", "plataforma_extraida", "porta_extraida",
                "tipo_evento_extraido", "dt_alarme", "dt_normalizacao"],
    "ZM01": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM02": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM03": ["Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada"],
}
for nome, df in bases.items():
    print(f"\n{nome}")
    perfil = pd.DataFrame({
        "coluna": df.columns,
        "tipo_lido": df.dtypes.astype(str).values,
        "ausentes": df.isna().sum().values,
        "ausentes_pct": df.isna().mean().mul(100).round(2).values,
        "strings_vazias": [df[c].map(lambda v: isinstance(v, str) and not v.strip()).sum()
                           if df[c].dtype == "object" else 0 for c in df.columns],
    })
    display(perfil)
    display(df[AMOSTRAS[nome]].head(5))



Alarmes


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,TAG,object,0,0.00,0
1,Horário do Alarme,object,0,0.00,0
2,Estação,object,0,0.00,0
3,Porta,float64,170,0.02,0
4,Alarme,object,0,0.00,0
5,Horário da Normalizacao,object,0,0.00,0
6,Tempo de Normalização,object,0,0.00,0
7,tempo_normalizacao_seg,int64,0,0.00,0
8,dt_alarme,object,0,0.00,0
9,dt_normalizacao,object,0,0.00,0


,TAG,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido,dt_alarme,dt_normalizacao
0,PSD_FAL_FAL2_PDM09_OBS,FAL,FAL2,9.0,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670
1,PSD_LUZ_LUZ1_PDM08_OBS,LUZ,LUZ1,8.0,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917
2,PSD_FAL_FAL1_PDM15_OBS,FAL,FAL1,15.0,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670
3,PSD_LUZ_LUZ1_PDM19_OBS,LUZ,LUZ1,19.0,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917
4,PSD_FRE_FRE1_PDM14_OBS,FRE,FRE1,14.0,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847



ZM01


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,1,0.01,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],326,3.09,0


...

,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,5,0.43,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],39,3.35,0


,nota,ordem,local_de_instalacao,inicio_avaria_datetime,fim_avaria_datetime
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56



ZM03


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,Ordem,int64,0,0.00,0
1,Tipo de ordem,object,0,0.00,0
2,Centro de trabalho,object,0,0.00,0
3,Local de instalação,object,0,0.00,0
4,Texto breve,object,0,0.00,0
5,Plano de manutenção,int64,0,0.00,0
6,Status usuário,object,0,0.00,0
7,Status do sistema,object,0,0.00,0
8,Data-base do início,object,0,0.00,0
9,Hr.início programada,object,0,0.00,0


,Ordem,Local de instalação,DT_Inicio,DT_Fim,cancelada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False


## 3. Datas e período dos registros

A conversão ocorre em séries auxiliares; valores inválidos são contabilizados. Mínimo e máximo são **datas de eventos registrados**, não confirmação de cobertura contínua. Meses sem registros também não demonstram ausência de falhas.

Em ZM01/ZM02, `inicio_avaria_datetime` é um campo candidato a início da avaria, não prova de execução da manutenção ou de quando a informação ficou disponível. Em ZM03, a inspeção compara `DT_Inicio`/`DT_Fim` com os campos de programação: equivalência não comprova execução.

In [4]:
DATAS = {
    "Alarmes": ["dt_alarme", "dt_normalizacao"],
    "ZM01": ["inicio_avaria_datetime", "fim_avaria_datetime", "analise_datetime", "encerramento_datetime"],
    "ZM02": ["inicio_avaria_datetime", "fim_avaria_datetime", "encerramento_datetime"],
    "ZM03": ["DT_Inicio", "DT_Fim", "Data-base do início", "Data-base do fim"],
}
datas = {}
resumo_datas = []
for nome, colunas in DATAS.items():
    for coluna in colunas:
        original = bases[nome][coluna]
        convertida = pd.to_datetime(original, errors="coerce", format="mixed")
        datas[nome, coluna] = convertida
        resumo_datas.append({
            "fonte": nome, "coluna": coluna, "menor_data": convertida.min(),
            "maior_data": convertida.max(), "ausentes_origem": original.isna().sum(),
            "nao_convertidas": (original.notna() & convertida.isna()).sum(),
        })
display(pd.DataFrame(resumo_datas))

PARES = {"Alarmes": ("dt_alarme", "dt_normalizacao"),
         "ZM01": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM02": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM03": ("DT_Inicio", "DT_Fim")}
for nome, (inicio, fim) in PARES.items():
    a, b = datas[nome, inicio], datas[nome, fim]
    print(f"{nome}: fim anterior ao início = {(b < a).sum()}; fim ausente/inválido = {b.isna().sum()}")

mensais = {}
for nome, coluna in {"Alarmes": "dt_alarme", "ZM01": "inicio_avaria_datetime",
                     "ZM02": "inicio_avaria_datetime", "ZM03": "DT_Inicio"}.items():
    serie = datas[nome, coluna].dropna()
    if not serie.empty:
        contagem = serie.dt.to_period("M").value_counts().sort_index()
        mensais[nome] = contagem.reindex(pd.period_range(contagem.index.min(), contagem.index.max(), freq="M"), fill_value=0)
print("Registros por mês (ausente fora do intervalo de eventos de cada fonte):")
display(pd.DataFrame(mensais).sort_index().astype("Int64"))

for derivada, base in [("DT_Inicio", "Data-base do início"), ("DT_Fim", "Data-base do fim")]:
    a, b = datas["ZM03", derivada], datas["ZM03", base]
    validos = a.notna() & b.notna()
    print(f"ZM03: {derivada} e {base}: {validos.sum()} pares válidos; "
          f"{(a[validos].dt.normalize() == b[validos].dt.normalize()).sum()} no mesmo dia.")
display(bases["ZM03"][["Data-base do início", "Hr.início programada", "DT_Inicio",
                       "Data-base do fim", "Fim-base (hora)", "DT_Fim"]].head())


,fonte,coluna,menor_data,maior_data,ausentes_origem,nao_convertidas
0,Alarmes,dt_alarme,2023-02-27 06:26:56.897,2026-05-12 08:16:52.657,0,0
1,Alarmes,dt_normalizacao,2023-02-27 06:27:11.897,2026-05-12 08:16:53.670,0,0
2,ZM01,inicio_avaria_datetime,2023-01-01 02:39:00.000,2026-04-01 22:43:00.000,0,0
3,ZM01,fim_avaria_datetime,2023-01-01 03:39:00.000,2026-04-01 23:25:57.000,326,0
4,ZM01,analise_datetime,2024-11-14 09:59:15.000,2026-05-06 13:39:20.000,5756,0
5,ZM01,encerramento_datetime,2023-01-01 05:16:11.000,2026-04-30 14:01:20.000,1,0
6,ZM02,inicio_avaria_datetime,2023-01-03 03:55:59.000,2026-03-28 14:15:00.000,0,0
7,ZM02,fim_avaria_datetime,2023-01-07 11:30:00.000,2026-05-14 02:00:00.000,39,0
8,ZM02,encerramento_datetime,2023-01-07 17:07:19.000,2026-05-14 04:33:52.000,7,0
9,ZM03,DT_Inicio,2022-12-26 05:30:00.000,2026-03-30 21:30:00.000,0,0


Alarmes: fim anterior ao início = 0; fim ausente/inválido = 0
ZM01: fim anterior ao início = 0; fim ausente/inválido = 326
ZM02: fim anterior ao início = 0; fim ausente/inválido = 39
ZM03: fim anterior ao início = 0; fim ausente/inválido = 0
Registros por mês (ausente fora do intervalo de eventos de cada fonte):


,Alarmes,ZM01,ZM02,ZM03
2022-12,<NA>,<NA>,<NA>,1
2023-01,<NA>,147,51,22
2023-02,50,171,29,16
2023-03,43,178,40,16
2023-04,0,282,31,19
2023-05,0,293,27,18
2023-06,0,292,35,15
2023-07,0,249,18,19
2023-08,0,310,7,12
2023-09,0,199,37,17


ZM03: DT_Inicio e Data-base do início: 569 pares válidos; 569 no mesmo dia.
ZM03: DT_Fim e Data-base do fim: 569 pares válidos; 567 no mesmo dia.


,Data-base do início,Hr.início programada,DT_Inicio,Data-base do fim,Fim-base (hora),DT_Fim
0,2026-03-23,21:30:00,2026-03-23 21:30:00,2026-03-24,04:30:00,2026-03-24 04:30:00
1,2026-03-24,21:30:00,2026-03-24 21:30:00,2026-03-25,04:30:00,2026-03-25 04:30:00
2,2026-03-17,21:30:00,2026-03-17 21:30:00,2026-03-18,04:30:00,2026-03-18 04:30:00
3,2026-03-16,21:30:00,2026-03-16 21:30:00,2026-03-17,04:30:00,2026-03-17 04:30:00
4,2026-03-30,21:30:00,2026-03-30 21:30:00,2026-04-02,04:30:00,2026-04-02 04:30:00


## 4. Identificação dos ativos — extração exploratória

As regras abaixo são deliberadamente conservadoras e criam tabelas auxiliares. Formatos desconhecidos ficam para revisão, sem inferência por texto livre.

- SAP: `SP04-6-PSD-BUT-PLT1-PP05PDM` → BUT / 1 / 05 / PDM.
- Alarme: `PSD_BUT_BUT1_PDM05_OBS` → BUT / 1 / 05 / PDM.
- Registros apenas de estação ou plataforma permanecem nesse nível.
- PDM, PFX e outros tipos não são unificados automaticamente.

A chave completa é provisória e inclui o tipo. Esta etapa mantém os eventos separados e preserva o nível de identificação das manutenções coletivas.

In [5]:
def extrair_ativos(serie, alarme=False):
    texto = serie.astype("string").str.strip().str.upper()
    if alarme:
        padrao = r"^PSD_(?P<estacao>[A-Z]{3})_(?P<estacao_plataforma>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_(?P<evento>[A-Z0-9]+)$"
        out = texto.str.extract(padrao)
        # Aceita PSD_P_BUT1_PDM07_FC e PSD_P_MBI1_PDM10FC.
        alternativo = texto.str.extract(
            r"^PSD_P_(?P<estacao>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_?(?P<evento>[A-Z][A-Z0-9]*)$")
        alternativo["estacao_plataforma"] = alternativo["estacao"]
        out = out.combine_first(alternativo)
        # Tags com estações divergentes não recebem chave válida.
        divergente = out.estacao.ne(out.estacao_plataforma).fillna(False)
        out.loc[divergente, ["estacao", "plataforma", "porta", "tipo_ativo"]] = pd.NA
    else:
        out = texto.str.extract(
            r"^SP\d+-\d+-PSD-(?P<estacao>[A-Z]{3})(?:-|$)")
        out["plataforma"] = texto.str.extract(r"-PLT(\d+)(?:-|$)", expand=False)
        porta = texto.str.extract(r"-PP(?P<porta>\d+)(?P<tipo_ativo>[A-Z]+)$")
        out = out.join(porta)
    out = out[["estacao", "plataforma", "porta", "tipo_ativo"]].copy()
    for coluna in ["plataforma", "porta"]:
        out[coluna] = pd.to_numeric(out[coluna], errors="coerce").astype("Int64").astype("string")
    completa = out.notna().all(axis=1)
    # Nível identificável não significa que um subativo é uma porta.
    out["nivel_identificavel"] = "não reconhecido"
    out.loc[out.estacao.notna(), "nivel_identificavel"] = "estação"
    out.loc[out.estacao.notna() & out.plataforma.notna(), "nivel_identificavel"] = "plataforma"
    out.loc[completa, "nivel_identificavel"] = "porta e tipo"
    out["chave_candidata"] = out.estacao + "|" + out.plataforma + "|" + out.porta + "|" + out.tipo_ativo
    out["identificacao_original"] = serie
    return out

COLUNA_ATIVO = {"Alarmes": "TAG", "ZM01": "local_de_instalacao",
                "ZM02": "local_de_instalacao", "ZM03": "Local de instalação"}
ativos = {nome: extrair_ativos(df[COLUNA_ATIVO[nome]], alarme=nome == "Alarmes")
          for nome, df in bases.items()}
for nome, df in ativos.items():
    print(f"\n{nome} — quantidade de registros por nível identificável")
    display(df.nivel_identificavel.value_counts().rename_axis("nível").to_frame("registros"))
    print("Exemplos de identificação (incluindo formatos que exigem revisão):")
    display(df.drop_duplicates("identificacao_original").groupby("nivel_identificavel", sort=False).head(4))
    print("Tipos identificados:")
    display(df.tipo_ativo.value_counts(dropna=False).to_frame("registros"))



Alarmes — quantidade de registros por nível identificável


,registros
nível,
porta e tipo,742336
não reconhecido,170


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,PSD_FAL_FAL2_PDM09_OBS
1,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,PSD_LUZ_LUZ1_PDM08_OBS
2,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,PSD_FAL_FAL1_PDM15_OBS
3,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,PSD_LUZ_LUZ1_PDM19_OBS
104431,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_LUZ1_CBTC_BIM_FAULT_Value


Tipos identificados:


...

,registros
nível,
plataforma,313
estação,173
não reconhecido,83


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,PTA,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PTA
1,FAL,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-FAL
2,PIN,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PIN
3,BUT,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-BUT
4,VSO,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-VSO-PLT1
9,MBI,2,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT2
10,MBI,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT1
11,PTA,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-PTA-PLT1
260,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,SP04-6-PSD


Tipos identificados:


,registros
tipo_ativo,
<NA>,569


## 5. Compatibilidade das chaves

A interseção abaixo mede apenas a existência da mesma chave nas fontes, **sem associação temporal ou causal**. Uma chave ausente pode decorrer de cobertura, nomenclatura ou falta de eventos. A TAG é comparada aos campos já extraídos dos alarmes para conferir consistência.

In [6]:
a = bases["Alarmes"]
extraido = ativos["Alarmes"]
comparacoes = {
    "estacao": a["estacao_extraida"].astype("string").str.upper().str.strip(),
    "plataforma": a["plataforma_extraida"].astype("string").str.extract(r"(\d+)$", expand=False),
    "porta": pd.to_numeric(a["porta_extraida"], errors="coerce").astype("Int64").astype("string"),
}
for campo, existente in comparacoes.items():
    if campo == "plataforma":
        existente = pd.to_numeric(existente, errors="coerce").astype("Int64").astype("string")
    validos = extraido[campo].notna() & existente.notna()
    print(f"Alarmes — {campo}: {validos.sum()} comparáveis, "
          f"{(extraido.loc[validos, campo] != existente[validos]).sum()} divergências.")

chaves_alarmes = set(extraido.chave_candidata.dropna())
resumo_chaves = []
for nome, df in ativos.items():
    chaves = set(df.chave_candidata.dropna())
    resumo_chaves.append({"fonte": nome, "chaves_completas_distintas": len(chaves),
                          "presentes_nos_alarmes": len(chaves & chaves_alarmes),
                          "ausentes_nos_alarmes": len(chaves - chaves_alarmes)})
display(pd.DataFrame(resumo_chaves))

todos = pd.concat([df[["estacao", "plataforma", "porta", "tipo_ativo"]].dropna().drop_duplicates()
                   for df in ativos.values()]).drop_duplicates()
colisoes = todos.groupby(["estacao", "plataforma", "porta"]).tipo_ativo.nunique()
print("Combinações estação/plataforma/número com mais de um tipo:", colisoes.gt(1).sum())
display(colisoes[colisoes.gt(1)].rename("quantidade_tipos").head(10).to_frame())


Alarmes — estacao: 742336 comparáveis, 0 divergências.


Alarmes — plataforma: 742336 comparáveis, 0 divergências.
Alarmes — porta: 742336 comparáveis, 0 divergências.


,fonte,chaves_completas_distintas,presentes_nos_alarmes,ausentes_nos_alarmes
0,Alarmes,978,978,0
1,ZM01,589,522,67
2,ZM02,518,345,173
3,ZM03,0,0,0


Combinações estação/plataforma/número com mais de um tipo: 475


quantidade_tipos
estacao plataforma porta                  
BUT     1          1                     2
                   10                    2
                   11                    2
                   12                    2
                   13                    2
                   15                    3
                   16                    2
                   17                    2
                   18                    3
                   19                    2

## 6. Duplicidades e flags

Repetição de ordem ou nota não implica automaticamente duplicata: pode representar atualização ou vários registros legítimos. A contagem inclui todas as linhas envolvidas (`keep=False`) e também o excedente de duplicatas exatas. Todos os registros são preservados.

In [7]:
resumo_duplicatas = []
for nome, df in bases.items():
    resumo_duplicatas.append({"fonte": nome,
                              "linhas_em_duplicatas_exatas": df.duplicated(keep=False).sum(),
                              "excedente_duplicatas_exatas": df.duplicated().sum()})
display(pd.DataFrame(resumo_duplicatas))
for nome, colunas in {"ZM01": ["ordem", "nota"], "ZM02": ["ordem", "nota"], "ZM03": ["Ordem"]}.items():
    for coluna in colunas:
        serie = bases[nome][coluna].dropna()
        print(f"{nome}/{coluna}: {serie.duplicated(keep=False).sum()} linhas com identificador repetido; "
              f"{serie[serie.duplicated(keep=False)].nunique()} identificadores repetidos.")

chave_evento = ["TAG", "dt_alarme"]
validos = a[chave_evento].notna().all(axis=1)
print("Alarmes: linhas com TAG + dt_alarme repetidos:",
      a.loc[validos].duplicated(chave_evento, keep=False).sum())
for nome, df in bases.items():
    flags = [c for c in df.columns if c.endswith("_flag") or c == "cancelada"]
    for coluna in flags:
        print(f"{nome} — {coluna}")
        display(df[coluna].value_counts(dropna=False).rename_axis("valor").to_frame("registros"))

# Apenas diagnóstico: não reinserir registros excluídos automaticamente.
excluidos = auditorias.get(("ZM01", "Outliers_removidos"))
if excluidos is not None:
    print(f"ZM01: {len(excluidos)} registros preservados em Outliers_removidos.")
    print("Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.")


,fonte,linhas_em_duplicatas_exatas,excedente_duplicatas_exatas
0,Alarmes,0,0
1,ZM01,0,0
2,ZM02,0,0
3,ZM03,0,0


ZM01/ordem: 136 linhas com identificador repetido; 66 identificadores repetidos.
ZM01/nota: 213 linhas com identificador repetido; 104 identificadores repetidos.
ZM02/ordem: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM02/nota: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM03/Ordem: 0 linhas com identificador repetido; 0 identificadores repetidos.


Alarmes: linhas com TAG + dt_alarme repetidos: 449829
ZM01 — indisponivel_flag


,registros
valor,
True,6972
False,3571


ZM01 — falha_flag


...

ZM02 — duracao_zero_flag


,registros
valor,
False,1158
True,5


ZM03 — cancelada


,registros
valor,
False,559
True,10


ZM01: 103 registros preservados em Outliers_removidos.
Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.


## 7. Decisões necessárias antes da tabela mestra

1. **Ativos:** validar os formatos não reconhecidos, os tipos elegíveis (por exemplo, PDM) e se tipo deve integrar a chave. Não misturar portas homônimas de plataformas diferentes.
2. **Manutenções coletivas:** definir como históricos de estação/plataforma entram como contexto, sem convertê-los em intervenções individuais comprovadas.
3. **Semântica temporal:** confirmar a data efetiva de cada manutenção e quando o registro ficou disponível. Em especial, não tratar datas programadas da ZM03 como prova de execução. Status atuais e textos atualizados depois do evento não reconstituem automaticamente o estado histórico.
4. **Falha:** definir critérios para episódios ZM01 e eventual evidência de defeitos em ZM02, revisar os registros excluídos na limpeza ZM01 e depois estabelecer a regra de associação com alarmes. ZM03 não é alvo positivo por si só.
5. **Cobertura:** confirmar os intervalos completos de cada fonte e do cadastro de ativos; não deduzir início de operação da primeira ocorrência nem cobertura pelo último evento.
6. **Duplicidades:** revisar ordens/notas repetidas antes de contar intervenções ou episódios.

**Resultado da inspeção:** inventário, perfis de qualidade, períodos de eventos e chaves candidatas em memória. As seções seguintes apresentam a preparação das tabelas e o calendário inicial. A definição do alvo permanece pendente.

## 8. Preparar as quatro tabelas — passo a passo

Esta seção prepara os dados para a tabela mestra com operações simples do pandas, sem novas funções.

A preparação reutiliza as identificações verificadas na inspeção (`ativos`). A chave candidata tem o formato `BUT|1|5|PDM`: estação BUT, plataforma 1, número 5, tipo PDM. O tipo permanece na chave para distinguir, por exemplo, PDM e MDU enquanto a relação entre eles não for confirmada.

- `.copy()`: cria uma cópia para trabalhar sem modificar a tabela lida.
- `.rename()`: troca nomes de colunas.
- `.join()`: acrescenta as identificações da inspeção à mesma linha de origem. **Não é um cruzamento entre alarmes e manutenções.**

As linhas sem chave completa continuam nas tabelas. Esta etapa preserva os registros e as datas ausentes; a definição do alvo permanece pendente.

In [8]:
# Colunas de identificação verificadas na inspeção.
colunas_identificacao = [
    "estacao", "plataforma", "porta", "tipo_ativo",
    "nivel_identificavel", "chave_candidata",
]


### 8.1 Alarmes

A seleção inclui a TAG, o tipo do evento e seus horários. `data_alarme` representa o dia da ocorrência, usado mais adiante para contar alarmes.

A normalização pode acontecer depois do dia do alarme. Sua duração final não pode ser usada numa previsão feita antes de o alarme normalizar.

In [9]:
alarmes = bases["Alarmes"][[
    "TAG", "tipo_evento_extraido", "dt_alarme", "dt_normalizacao",
]].copy()

alarmes = alarmes.rename(columns={
    "TAG": "tag",
    "tipo_evento_extraido": "tipo_evento",
    "dt_alarme": "inicio_alarme",
    "dt_normalizacao": "fim_alarme",
})

alarmes = alarmes.join(ativos["Alarmes"][colunas_identificacao])

alarmes["inicio_alarme"] = pd.to_datetime(alarmes["inicio_alarme"], errors="coerce")
alarmes["fim_alarme"] = pd.to_datetime(alarmes["fim_alarme"], errors="coerce")
alarmes["data_alarme"] = alarmes["inicio_alarme"].dt.normalize()

display(alarmes.head())


,tag,tipo_evento,inicio_alarme,fim_alarme,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_alarme
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,2026-05-12
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,2026-05-12
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,2026-05-12
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,2026-05-12
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847,FRE,1,14,PDM,porta e tipo,FRE|1|14|PDM,2026-05-12


### 8.2 ZM01 — corretiva emergencial

A tabela mantém nota, ordem, local e início/fim da avaria. `data_avaria` é uma candidata à data de falha, mas os critérios do alvo e os 103 registros separados na limpeza ainda precisam de revisão. `falha_flag` é preservada para essa revisão; não será usada automaticamente como informação histórica disponível no dia.

In [10]:
zm01 = bases["ZM01"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime", "falha_flag",
]].copy()

zm01 = zm01.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm01 = zm01.join(ativos["ZM01"][colunas_identificacao])

zm01["inicio_avaria"] = pd.to_datetime(zm01["inicio_avaria"], errors="coerce")
zm01["fim_avaria"] = pd.to_datetime(zm01["fim_avaria"], errors="coerce")
zm01["data_avaria"] = zm01["inicio_avaria"].dt.normalize()

display(zm01.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,falha_flag,...,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,14943931,53954586.0,SP04-6-PSD-BUT,2025-11-14 06:35:28,NaT,True,...,<NA>,<NA>,<NA>,estação,<NA>,2025-11-14
1,15133619,54145839.0,SP04-6-PSD-BUT,2026-02-27 07:33:00,2026-02-27 12:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2026-02-27
2,13894133,52913343.0,SP04-6-PSD-BUT-P1S1,2024-04-18 06:00:00,2024-04-18 07:30:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2024-04-18
3,14713349,53724958.0,SP04-6-PSD-BUT-P1S1,2025-07-07 18:38:00,2025-07-08 03:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-07-07
4,14643768,53655728.0,SP04-6-PSD-BUT-P1S1-PSSE1,2025-05-28 09:10:00,2025-05-28 10:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-05-28


### 8.3 ZM02 — corretiva programada

A tabela preserva o nome `inicio_avaria`, conforme a identificação do campo na fonte. Sua interpretação como data de execução da manutenção exige confirmação. Um registro ZM02 não é classificado automaticamente como falha.

In [11]:
zm02 = bases["ZM02"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime",
]].copy()

zm02 = zm02.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm02 = zm02.join(ativos["ZM02"][colunas_identificacao])

zm02["inicio_avaria"] = pd.to_datetime(zm02["inicio_avaria"], errors="coerce")
zm02["fim_avaria"] = pd.to_datetime(zm02["fim_avaria"], errors="coerce")
zm02["data_avaria"] = zm02["inicio_avaria"].dt.normalize()

display(zm02.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-01-25
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-05-30
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-06-19
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-09-30
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56,BUT,1,<NA>,<NA>,plataforma,<NA>,2023-07-21


### 8.4 ZM03 — preventiva

Esta preparação trata `DT_Inicio` e `DT_Fim` como referências de programação, sujeitas à confirmação. O status `cancelada` é mantido para revisão: uma ordem não cancelada não comprova execução.

A inspeção não encontrou identificação individual completa de porta nessa base. Por isso, a preparação mantém as preventivas no nível de identificação disponível, sem atribuí-las a cada porta. Os registros sem estação reconhecida também permanecem disponíveis para revisão.

In [12]:
zm03 = bases["ZM03"][[
    "Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada",
]].copy()

zm03 = zm03.rename(columns={
    "Ordem": "ordem",
    "Local de instalação": "local_original",
    "DT_Inicio": "inicio_programado",
    "DT_Fim": "fim_programado",
})

zm03 = zm03.join(ativos["ZM03"][colunas_identificacao])

zm03["inicio_programado"] = pd.to_datetime(zm03["inicio_programado"], errors="coerce")
zm03["fim_programado"] = pd.to_datetime(zm03["fim_programado"], errors="coerce")
zm03["data_programada"] = zm03["inicio_programado"].dt.normalize()

display(zm03.head())


,ordem,local_original,inicio_programado,fim_programado,cancelada,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_programada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False,PTA,<NA>,<NA>,<NA>,estação,<NA>,2026-03-23
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False,FAL,<NA>,<NA>,<NA>,estação,<NA>,2026-03-24
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False,PIN,<NA>,<NA>,<NA>,estação,<NA>,2026-03-17
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-03-16
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False,VSO,1,<NA>,<NA>,plataforma,<NA>,2026-03-30


## 9. Conferência das identificações

Este resumo conta **registros**, não portas distintas ou falhas. A identificação completa é candidata: ainda depende da escolha dos tipos de ativo que entrarão no modelo.

O pequeno `for` repete a mesma conferência nas quatro tabelas. Os `assert` verificam se todas as linhas e suas posições foram preservadas.

In [13]:
tabelas_preparadas = {"Alarmes": alarmes, "ZM01": zm01, "ZM02": zm02, "ZM03": zm03}
resumo = []

for nome, tabela in tabelas_preparadas.items():
    assert len(tabela) == len(bases[nome]), f"A quantidade de linhas mudou em {nome}."
    assert tabela.index.equals(bases[nome].index), f"A ordem das linhas mudou em {nome}."

    resumo.append({
        "fonte": nome,
        "total_registros": len(tabela),
        "com_chave_completa": tabela["chave_candidata"].notna().sum(),
        "sem_chave_completa": tabela["chave_candidata"].isna().sum(),
    })

display(pd.DataFrame(resumo))


,fonte,total_registros,com_chave_completa,sem_chave_completa
0,Alarmes,742506,742336,170
1,ZM01,10543,9357,1186
2,ZM02,1163,1010,153
3,ZM03,569,0,569


## 10. Primeira contagem diária: alarmes

Esta etapa conta os **registros de alarmes por chave candidata e dia**, com `groupby()` e `size()`.

Por enquanto, a contagem mantém registros repetidos da fonte. Ela é exploratória: não representa episódios únicos e ainda não é a variável final `qtd_alarmes_1d`. As duplicidades e os tipos de ativo precisam de revisão antes da modelagem.

Linhas sem chave ou data ficam em `alarmes_para_revisar`. Não serão atribuídas a uma porta por aproximação. A tabela diária contém somente dias com registros; dias ausentes permanecem sem preenchimento enquanto a cobertura não for confirmada.

In [14]:
# A contagem por ativo exige identificação completa e data.
pode_agrupar = alarmes["chave_candidata"].notna() & alarmes["data_alarme"].notna()

alarmes_identificados = alarmes.loc[pode_agrupar].copy()
alarmes_para_revisar = alarmes.loc[~pode_agrupar].copy()

alarmes_diarios = (
    alarmes_identificados
    .groupby(["chave_candidata", "data_alarme"])
    .size()
    .reset_index(name="qtd_registros_alarmes")
)

# A conferência verifica se houve perda ou contagem duplicada no agrupamento.
assert len(alarmes_identificados) + len(alarmes_para_revisar) == len(alarmes)
assert alarmes_diarios["qtd_registros_alarmes"].sum() == len(alarmes_identificados)
assert not alarmes_diarios.duplicated(["chave_candidata", "data_alarme"]).any()

print("Registros usados na contagem:", len(alarmes_identificados))
print("Registros separados para revisão:", len(alarmes_para_revisar))
display(alarmes_diarios.head(10))


Registros usados na contagem: 742336
Registros separados para revisão: 170


,chave_candidata,data_alarme,qtd_registros_alarmes
0,BUT|1|10|MDU,2023-11-12,1
1,BUT|1|10|MDU,2023-11-13,1
2,BUT|1|10|MDU,2023-11-27,2
3,BUT|1|10|MDU,2023-12-06,11
4,BUT|1|10|MDU,2023-12-31,5
5,BUT|1|10|MDU,2024-01-01,3
6,BUT|1|10|MDU,2024-01-03,4
7,BUT|1|10|MDU,2024-01-08,19
8,BUT|1|10|MDU,2024-01-14,31
9,BUT|1|10|MDU,2024-01-20,5


### Resultado desta etapa

A preparação produz quatro tabelas em memória (`alarmes`, `zm01`, `zm02`, `zm03`) e uma primeira contagem diária (`alarmes_diarios`). Os arquivos de origem continuam intactos e nenhum CSV foi exportado.

**Continuação:** as próximas seções adotam um tipo de ativo e um recorte exploratório para construir o calendário diário e juntar as contagens de alarmes. A confirmação de cobertura, o alvo `falha_7d`, as janelas móveis e o treinamento permanecem para etapas posteriores.

## 11. Tipo de ativo e período inicial

O primeiro calendário considera **PDM**, sem agregar MDU ou outros tipos. Essa escolha inicial pode ser alterada na célula de configuração.

A leitura das TAGs também reconhece o formato de MBI sem separador entre número e evento, como `PSD_P_MBI1_PDM10FC`. Assim, esses registros passam pela mesma identificação dos demais alarmes.

O período exploratório vai de **17/10/2023 a 28/03/2026**. O início acompanha a retomada de registros em várias estações após meses vazios em 2023. O fim evita avançar além da última data de início de avaria registrada em ZM02, a mais antiga entre os limites finais dos quatro campos de referência. Essas datas **não comprovam cobertura completa**: servem apenas para construir e examinar a estrutura da tabela.

O resumo por estação ajuda a visualizar diferenças entre os períodos registrados. Primeiro e último alarme não representam início e fim de operação.

In [15]:
# Configuração simples do recorte exploratório.
TIPO_ATIVO = "PDM"
DATA_INICIO = pd.Timestamp("2023-10-17")
DATA_FIM = pd.Timestamp("2026-03-28")

assert DATA_INICIO <= DATA_FIM, "O início precisa ser anterior ou igual ao fim."

alarmes_pdm = alarmes.loc[alarmes["tipo_ativo"].eq(TIPO_ATIVO)].copy()

periodo_por_estacao = (
    alarmes_pdm.groupby("estacao")["data_alarme"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "primeiro_registro", "max": "ultimo_registro",
                     "count": "qtd_registros"})
)

display(periodo_por_estacao)


,primeiro_registro,ultimo_registro,qtd_registros
estacao,,,
BUT,2023-10-17,2026-05-12,84434
FAL,2023-10-17,2026-05-12,25011
FRA,2023-10-18,2026-04-14,145188
FRE,2023-02-27,2026-05-12,2111
HIG,2023-10-17,2026-05-12,85184
LUZ,2023-10-17,2026-05-12,39148
MBI,2024-06-05,2025-12-31,72913
PIN,2023-10-17,2025-12-31,114359
PTA,2023-10-24,2025-01-26,29048


## 12. Lista das portas presentes no recorte

A lista reúne identificações completas de PDM encontradas nos alarmes, na ZM01 ou na ZM02 durante o período escolhido. A inclusão das corretivas evita limitar a lista apenas às portas com alarmes.

- `.between()` seleciona datas entre os dois limites, incluindo ambos.
- `pd.concat()` reúne as listas.
- `.drop_duplicates()` mantém uma linha por identificação.

Esta é uma **lista exploratória de ativos registrados**, não um cadastro de portas em operação. O calendário pode conter datas anteriores à primeira identificação de uma porta. A definição de amostras de treino exigirá validar o cadastro e a cobertura, sem usar uma identificação futura como prova de operação no passado. A ZM03 permanece fora desta lista por não identificar portas individualmente.

In [16]:
colunas_porta = ["chave_candidata", "estacao", "plataforma", "porta", "tipo_ativo"]

portas_alarmes = alarmes_pdm.loc[
    alarmes_pdm["data_alarme"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm01 = zm01.loc[
    zm01["tipo_ativo"].eq(TIPO_ATIVO)
    & zm01["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm02 = zm02.loc[
    zm02["tipo_ativo"].eq(TIPO_ATIVO)
    & zm02["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas = pd.concat([portas_alarmes, portas_zm01, portas_zm02], ignore_index=True)
portas = portas.dropna(subset=colunas_porta)
portas = portas.drop_duplicates().sort_values("chave_candidata").reset_index(drop=True)

assert not portas.empty, "Nenhuma porta foi encontrada no recorte."
assert not portas["chave_candidata"].duplicated().any(), "Há identificações conflitantes."

print("Quantidade de portas na lista:", len(portas))
display(portas.head(10))


Quantidade de portas na lista: 528


,chave_candidata,estacao,plataforma,porta,tipo_ativo
0,BUT|1|10|PDM,BUT,1,10,PDM
1,BUT|1|11|PDM,BUT,1,11,PDM
2,BUT|1|12|PDM,BUT,1,12,PDM
3,BUT|1|13|PDM,BUT,1,13,PDM
4,BUT|1|14|PDM,BUT,1,14,PDM
5,BUT|1|15|PDM,BUT,1,15,PDM
6,BUT|1|16|PDM,BUT,1,16,PDM
7,BUT|1|17|PDM,BUT,1,17,PDM
8,BUT|1|18|PDM,BUT,1,18,PDM
9,BUT|1|19|PDM,BUT,1,19,PDM


## 13. Calendário: uma linha por porta e dia

`pd.date_range()` cria todos os dias do período. O `merge(how="cross")` combina cada porta com cada dia.

Por exemplo, uma lista com 2 portas e um calendário de 3 dias produz 6 linhas. A tabela ainda é uma estrutura exploratória; a existência de uma linha não significa que houve coleta de dados naquele dia.

In [17]:
calendario = pd.DataFrame({
    "data": pd.date_range(DATA_INICIO, DATA_FIM, freq="D")
})

grade_diaria = portas.merge(calendario, how="cross")

assert len(grade_diaria) == len(portas) * len(calendario)
assert not grade_diaria.duplicated(["chave_candidata", "data"]).any()

print("Portas:", len(portas))
print("Dias:", len(calendario))
print("Linhas do calendário:", len(grade_diaria))
display(grade_diaria.head())


Portas: 528
Dias: 894
Linhas do calendário: 472032


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21


## 14. Inclusão da contagem diária de alarmes

O `merge(how="left")` mantém todas as linhas do calendário e procura a contagem correspondente à mesma chave e data. `validate="one_to_one"` verifica se cada combinação aparece no máximo uma vez em cada tabela.

A coluna `qtd_registros_alarmes` mantém valores ausentes (`<NA>`) quando não há correspondência. O preenchimento com zero depende de confirmar que a coleta estava completa. A coluna `tem_registro_alarme` indica apenas a presença de registros; `False` não comprova ausência de alarmes.

As contagens ainda incluem possíveis repetições da fonte. Esta versão não está pronta para treinamento.

In [18]:
contagens_recorte = alarmes_diarios.loc[
    alarmes_diarios["chave_candidata"].isin(portas["chave_candidata"])
    & alarmes_diarios["data_alarme"].between(DATA_INICIO, DATA_FIM)
].copy()

contagens_recorte = contagens_recorte.rename(columns={"data_alarme": "data"})

tabela_mestra_inicial = grade_diaria.merge(
    contagens_recorte,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

# Int64, com I maiúsculo, permite números inteiros e valores ausentes.
tabela_mestra_inicial["qtd_registros_alarmes"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].astype("Int64")
)
tabela_mestra_inicial["tem_registro_alarme"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].notna()
)

display(tabela_mestra_inicial.head(10))


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17,<NA>,False
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18,<NA>,False
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19,<NA>,False
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20,<NA>,False
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21,<NA>,False
5,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-22,<NA>,False
6,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-23,<NA>,False
7,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-24,<NA>,False
8,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-25,<NA>,False
9,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-26,<NA>,False


## 15. Conferência da tabela inicial

As verificações conferem o tamanho do calendário, a unicidade de porta/dia e a conservação das contagens. O exemplo exibe dias de uma porta com registros no recorte, incluindo dias sem correspondência quando existirem.

In [19]:
assert len(tabela_mestra_inicial) == len(grade_diaria)
assert not tabela_mestra_inicial.duplicated(["chave_candidata", "data"]).any()
assert (
    tabela_mestra_inicial["qtd_registros_alarmes"].sum()
    == contagens_recorte["qtd_registros_alarmes"].sum()
)

print("Linhas da tabela inicial:", len(tabela_mestra_inicial))
print("Linhas com registros de alarmes:",
      tabela_mestra_inicial["tem_registro_alarme"].sum())
print("Linhas sem correspondência nos alarmes:",
      tabela_mestra_inicial["qtd_registros_alarmes"].isna().sum())

if not contagens_recorte.empty:
    primeiro_registro = contagens_recorte.iloc[0]
    exemplo = tabela_mestra_inicial.loc[
        tabela_mestra_inicial["chave_candidata"].eq(primeiro_registro["chave_candidata"])
        & tabela_mestra_inicial["data"].between(
            primeiro_registro["data"] - pd.Timedelta(days=3),
            primeiro_registro["data"] + pd.Timedelta(days=6),
        )
    ]
    display(exemplo)


Linhas da tabela inicial: 472032
Linhas com registros de alarmes: 47518
Linhas sem correspondência nos alarmes: 424514


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
21,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-07,<NA>,False
22,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-08,<NA>,False
23,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-09,<NA>,False
24,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-10,3,True
25,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-11,3,True
26,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-12,1,True
27,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-13,10,True
28,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-14,<NA>,False
29,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-15,<NA>,False
30,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-16,<NA>,False


### Resultado e próxima etapa

A variável `tabela_mestra_inicial` contém uma linha por PDM e dia, com identificação e contagem de registros de alarmes. O resultado permanece em memória; nenhum CSV é exportado.

A próxima etapa deverá definir a contagem de registros de manutenção, tratando ordens repetidas e o nível de identificação de cada ocorrência. ZM01/ZM02 com identificação completa poderão ser associadas à mesma porta. Registros coletivos e ZM03 precisarão de regras próprias de contexto. A formação do alvo exigirá também a revisão dos registros ZM01 excluídos na limpeza e da cobertura das fontes.

O calendário inicial não confirma operação diária, coleta completa ou ausência de falhas. Essas condições precisam ser resolvidas antes do preenchimento de zeros, das janelas de 30 dias e do alvo de sete dias.

## 16. Regra de contagem das corretivas

Esta etapa acrescenta informações de ZM01 e ZM02 ao calendário de PDM. A unidade contada é a **ordem por porta e dia de início da avaria**. Essa contagem ainda não representa episódios de falha nem manutenções executadas.

Uma ordem repetida com o mesmo local e dia entra apenas uma vez na contagem. Quando a mesma ordem apresenta locais ou dias diferentes, seus registros ficam separados para revisão. Registros sem número de ordem também ficam separados; a nota é preservada na tabela de origem.

A identificação de divergências utiliza a base disponível inteira apenas para auditoria. Ela não representa o conhecimento que existia em cada dia histórico. Antes da modelagem, a versão e a disponibilidade temporal dos registros precisarão ser confirmadas.

O laço `for` aplica a mesma conferência às duas tabelas. `nunique(dropna=False)` conta valores distintos, incluindo ausências, para não esconder divergências.

In [20]:
corretivas = {"ZM01": zm01, "ZM02": zm02}
corretivas_para_revisar = {}
corretivas_sem_ambiguidade = {}
resumo_ordens = []

for nome, tabela in corretivas.items():
    variacoes = tabela.groupby("ordem")[["local_original", "data_avaria"]].nunique(dropna=False)
    ordens_divergentes = variacoes.index[
        (variacoes["local_original"] > 1) | (variacoes["data_avaria"] > 1)
    ]

    precisa_revisao = tabela["ordem"].isna() | tabela["ordem"].isin(ordens_divergentes)
    corretivas_para_revisar[nome] = tabela.loc[precisa_revisao].copy()
    corretivas_sem_ambiguidade[nome] = tabela.loc[~precisa_revisao].copy()

    assert len(corretivas_para_revisar[nome]) + len(corretivas_sem_ambiguidade[nome]) == len(tabela)

    resumo_ordens.append({
        "fonte": nome,
        "linhas_originais": len(tabela),
        "linhas_sem_ordem": tabela["ordem"].isna().sum(),
        "ordens_com_local_ou_dia_divergente": len(ordens_divergentes),
        "linhas_para_revisar": precisa_revisao.sum(),
    })

display(pd.DataFrame(resumo_ordens))
display(corretivas_para_revisar["ZM01"][[
    "ordem", "nota", "local_original", "data_avaria",
]].head(10))


,fonte,linhas_originais,linhas_sem_ordem,ordens_com_local_ou_dia_divergente,linhas_para_revisar
0,ZM01,10543,1,0,1
1,ZM02,1163,5,0,5


,ordem,nota,local_original,data_avaria
5263,NaN,14445462,SP04-6-PSD-MBI-PLT1-PP11PDM,2025-02-07


## 17. Contagens de ZM01 e ZM02 por porta e dia

A seleção considera identificações presentes na lista `portas` e datas dentro do recorte. Os demais tipos, registros coletivos e datas externas permanecem nas tabelas anteriores.

`.drop_duplicates()` gera uma referência por ordem/local/dia **somente para a contagem**. A operação não escolhe qual versão de texto, status ou diagnóstico da ordem está correta.

As colunas finais mantêm o termo `inicio_avaria` para distinguir essa referência de uma data de execução.

In [21]:
corretivas_no_recorte = {}
contagens_corretivas = {}
resumo_corretivas = []

for nome, tabela in corretivas_sem_ambiguidade.items():
    dentro_do_recorte = (
        tabela["chave_candidata"].isin(portas["chave_candidata"])
        & tabela["data_avaria"].between(DATA_INICIO, DATA_FIM)
    )
    selecionadas = tabela.loc[dentro_do_recorte].copy()
    ordens_unicas = selecionadas.drop_duplicates(["ordem", "local_original", "data_avaria"])
    corretivas_no_recorte[nome] = ordens_unicas

    contagem = (
        ordens_unicas.groupby(["chave_candidata", "data_avaria"])
        .size()
        .reset_index(name="qtd_ordens")
        .rename(columns={"data_avaria": "data"})
    )
    contagens_corretivas[nome] = contagem

    assert contagem["qtd_ordens"].sum() == len(ordens_unicas)
    resumo_corretivas.append({
        "fonte": nome,
        "linhas_selecionadas": len(selecionadas),
        "repeticoes_na_contagem": len(selecionadas) - len(ordens_unicas),
        "ordens_contadas": len(ordens_unicas),
    })

zm01_diaria = contagens_corretivas["ZM01"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm01_inicio_avaria",
})
zm02_diaria = contagens_corretivas["ZM02"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm02_inicio_avaria",
})

display(pd.DataFrame(resumo_corretivas))
display(zm01_diaria.head())
display(zm02_diaria.head())


,fonte,linhas_selecionadas,repeticoes_na_contagem,ordens_contadas
0,ZM01,7256,42,7214
1,ZM02,583,1,582


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria
0,BUT|1|10|PDM,2023-11-10,2
1,BUT|1|10|PDM,2023-11-12,1
2,BUT|1|10|PDM,2023-11-13,2
3,BUT|1|10|PDM,2023-12-09,1
4,BUT|1|10|PDM,2023-12-20,1


,chave_candidata,data,qtd_ordens_zm02_inicio_avaria
0,BUT|1|10|PDM,2023-12-09,1
1,BUT|1|15|PDM,2025-08-07,1
2,BUT|1|18|PDM,2024-07-08,1
3,BUT|1|19|PDM,2024-12-04,1
4,BUT|1|23|PDM,2024-09-21,1


## 18. Junção das corretivas ao calendário

Cada junção usa a mesma porta e data. O `how="left"` preserva todas as linhas do calendário. O `validate="one_to_one"` impede a multiplicação de linhas caso uma contagem tenha chaves repetidas.

Valores sem correspondência permanecem ausentes. Em particular, a presença de ordens separadas para revisão e a cobertura ainda não confirmada impedem interpretar essas ausências automaticamente como zero falhas.

In [22]:
tabela_mestra_manutencoes = tabela_mestra_inicial.merge(
    zm01_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm02_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

colunas_corretivas = [
    "qtd_ordens_zm01_inicio_avaria",
    "qtd_ordens_zm02_inicio_avaria",
]
tabela_mestra_manutencoes[colunas_corretivas] = (
    tabela_mestra_manutencoes[colunas_corretivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert tabela_mestra_manutencoes["qtd_ordens_zm01_inicio_avaria"].sum() == zm01_diaria["qtd_ordens_zm01_inicio_avaria"].sum()
assert tabela_mestra_manutencoes["qtd_ordens_zm02_inicio_avaria"].sum() == zm02_diaria["qtd_ordens_zm02_inicio_avaria"].sum()

display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_corretivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_corretivas],
].head(10))


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria,qtd_ordens_zm02_inicio_avaria
24,BUT|1|10|PDM,2023-11-10,2,<NA>
26,BUT|1|10|PDM,2023-11-12,1,<NA>
27,BUT|1|10|PDM,2023-11-13,2,<NA>
53,BUT|1|10|PDM,2023-12-09,1,1
64,BUT|1|10|PDM,2023-12-20,1,<NA>
304,BUT|1|10|PDM,2024-08-16,1,<NA>
323,BUT|1|10|PDM,2024-09-04,2,<NA>
371,BUT|1|10|PDM,2024-10-22,1,<NA>
407,BUT|1|10|PDM,2024-11-27,1,<NA>
494,BUT|1|10|PDM,2025-02-22,1,<NA>


## 19. ZM03: separação por estação e plataforma

A programação preventiva será tratada como **contexto coletivo**, sem indicar uma intervenção comprovada em cada porta.

A seleção reconhece somente os locais completos nos formatos:

- `SP04-6-PSD-BUT`: contexto da estação.
- `SP04-6-PSD-BUT-PLT1`: contexto da plataforma.

Um local genérico como `SP04-6-PSD` fica separado. A correspondência exata evita tratar um subcomponente como manutenção de toda a estação.

Nesta base, cada ordem ZM03 aparece uma vez. A verificação interrompe a contagem se uma versão futura contiver ordens repetidas, ausentes ou status de cancelamento desconhecidos. A coluna `cancelada` descreve o cadastro disponível: seu valor não comprova o status conhecido na data programada.

In [23]:
assert zm03["ordem"].notna().all(), "Há ordens ZM03 sem identificação."
assert not zm03["ordem"].duplicated().any(), "As ordens ZM03 repetidas precisam de revisão."
assert zm03["cancelada"].isin([True, False]).all(), "Há status de cancelamento desconhecido."

locais_zm03 = zm03["local_original"].astype("string").str.strip().str.upper()

local_estacao = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}", na=False)
local_plataforma = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}-PLT\d+", na=False)
data_no_recorte = zm03["data_programada"].between(DATA_INICIO, DATA_FIM)

zm03_estacao = zm03.loc[local_estacao & data_no_recorte].copy()
zm03_plataforma = zm03.loc[local_plataforma & data_no_recorte].copy()
zm03_sem_escopo = zm03.loc[~(local_estacao | local_plataforma)].copy()

print("Ordens de estação no recorte:", len(zm03_estacao))
print("Ordens de plataforma no recorte:", len(zm03_plataforma))
print("Ordens sem escopo reconhecido na base completa:", len(zm03_sem_escopo))
display(zm03_sem_escopo[["ordem", "local_original", "data_programada"]].head())


Ordens de estação no recorte: 128
Ordens de plataforma no recorte: 230
Ordens sem escopo reconhecido na base completa: 83


,ordem,local_original,data_programada
260,52907929,SP04-6-PSD,2024-07-29
261,52907928,SP04-6-PSD,2024-07-15
262,52907927,SP04-6-PSD,2024-07-08
263,52907926,SP04-6-PSD,2024-07-01
264,52907925,SP04-6-PSD,2024-06-24


## 20. Contagens da programação preventiva

A agregação separa ordens canceladas e não canceladas na versão disponível. **Não cancelada não significa executada.** As quatro colunas são de auditoria da programação e ainda não são entradas aprovadas para o modelo.

Uma mesma ordem de estação não entra também na contagem de plataforma. As duas tabelas preservam os níveis originais.

In [24]:
# Conversão explícita para booleano após a conferência do status.
zm03_estacao["cancelada"] = zm03_estacao["cancelada"].astype(bool)
zm03_estacao["nao_cancelada"] = ~zm03_estacao["cancelada"]

zm03_estacao_diaria = (
    zm03_estacao.groupby(["estacao", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_estacao=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_estacao=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

zm03_plataforma["cancelada"] = zm03_plataforma["cancelada"].astype(bool)
zm03_plataforma["nao_cancelada"] = ~zm03_plataforma["cancelada"]

zm03_plataforma_diaria = (
    zm03_plataforma.groupby(["estacao", "plataforma", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_plataforma=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_plataforma=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

assert zm03_estacao_diaria.filter(like="qtd_").sum().sum() == len(zm03_estacao)
assert zm03_plataforma_diaria.filter(like="qtd_").sum().sum() == len(zm03_plataforma)

display(zm03_estacao_diaria.head())
display(zm03_plataforma_diaria.head())


,estacao,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao
0,BUT,2023-10-30,1,0
1,BUT,2023-11-26,1,0
2,BUT,2023-12-28,1,0
3,BUT,2024-01-24,1,0
4,BUT,2024-02-23,1,0


,estacao,plataforma,data,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
0,BUT,1,2024-01-02,1,0
1,BUT,1,2024-03-18,1,0
2,BUT,1,2024-06-03,1,0
3,BUT,1,2024-09-09,1,0
4,BUT,1,2024-12-05,1,0


## 21. Inclusão do contexto de ZM03

O contexto de estação é associado por estação e data. O de plataforma é associado por estação, plataforma e data. `validate="many_to_one"` permite que várias portas recebam o mesmo contexto, mas exige uma única linha na tabela coletiva para cada chave.

Por exemplo, uma ordem programada para BUT/PLT1 aparece como contexto nas linhas das PDM dessa plataforma naquele dia. Isso não representa uma ordem executada em cada PDM. Por esse motivo, essas colunas **não devem ser somadas entre portas** para obter o total de ordens.

As tabelas agregadas anteriores são a referência para totais de programação. Registros de estações ou plataformas sem PDM na lista continuam disponíveis nessas tabelas.

In [25]:
tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_estacao_diaria,
    on=["estacao", "data"],
    how="left",
    validate="many_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_plataforma_diaria,
    on=["estacao", "plataforma", "data"],
    how="left",
    validate="many_to_one",
)

colunas_preventivas = [
    "qtd_zm03_nao_canceladas_estacao",
    "qtd_zm03_canceladas_estacao",
    "qtd_zm03_nao_canceladas_plataforma",
    "qtd_zm03_canceladas_plataforma",
]
tabela_mestra_manutencoes[colunas_preventivas] = (
    tabela_mestra_manutencoes[colunas_preventivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert not tabela_mestra_manutencoes.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_manutencoes["qtd_registros_alarmes"].equals(
    tabela_mestra_inicial["qtd_registros_alarmes"]
)

print("Linhas após as quatro junções:", len(tabela_mestra_manutencoes))
print("Colunas adicionadas:", len(colunas_corretivas + colunas_preventivas))
display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_preventivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_preventivas],
].head(10))


Linhas após as quatro junções: 472032
Colunas adicionadas: 6


,chave_candidata,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
13,BUT|1|10|PDM,2023-10-30,1,0,<NA>,<NA>
40,BUT|1|10|PDM,2023-11-26,1,0,<NA>,<NA>
72,BUT|1|10|PDM,2023-12-28,1,0,<NA>,<NA>
77,BUT|1|10|PDM,2024-01-02,<NA>,<NA>,1,0
99,BUT|1|10|PDM,2024-01-24,1,0,<NA>,<NA>
129,BUT|1|10|PDM,2024-02-23,1,0,<NA>,<NA>
153,BUT|1|10|PDM,2024-03-18,1,0,1,0
183,BUT|1|10|PDM,2024-04-17,1,0,<NA>,<NA>
230,BUT|1|10|PDM,2024-06-03,<NA>,<NA>,1,0
326,BUT|1|10|PDM,2024-09-07,1,0,<NA>,<NA>


### Resultado da integração de manutenções

A variável `tabela_mestra_manutencoes` reúne o calendário de PDM, as contagens de registros de alarmes, as ordens ZM01/ZM02 por início da avaria e o contexto de programação ZM03. O resultado permanece em memória e os arquivos de origem permanecem intactos.

As tabelas `corretivas_para_revisar`, `zm03_sem_escopo` e a aba `Outliers_removidos` de ZM01 preservam casos que exigem análise antes da definição do alvo. As bases anteriores também mantêm registros coletivos e outros tipos de ativo que não entraram nas contagens individuais.

**Próximo passo:** revisar o critério de falha ZM01, os registros separados e o período de cobertura. Essas decisões permitem distinguir dias sem falha de dias sem informação. Somente depois poderão ser criados `falha_no_dia`, `falha_7d` e as variáveis históricas. Datas programadas de ZM03 e status atuais não serão usados automaticamente como histórico de execução.

## 22. Significado de `falha_flag`

O notebook de limpeza `eda/cleaning/limpeza-dados-ZM01.ipynb` transforma a marcação SAP `X` em `True`. Campos vazios são transformados em `False`. Portanto, **False significa ausência da marcação, não comprovação de ausência de falha**.

A comparação abaixo mantém dois critérios candidatos: toda ZM01 individual identificada e somente ZM01 com marcação de falha. Nenhum critério é adotado como alvo definitivo nesta etapa. Alarmes associados poderão complementar a análise, mas a falta de um alarme correspondente não invalida automaticamente uma ZM01.

In [26]:
resumo_flags = pd.DataFrame({
    "base_principal": bases["ZM01"]["falha_flag"].value_counts(dropna=False),
    "outliers_separados": auditorias[("ZM01", "Outliers_removidos")]["falha_flag"].value_counts(dropna=False),
}).rename_axis("marcacao_falha_sap")

display(resumo_flags)


,base_principal,outliers_separados
marcacao_falha_sap,,
False,7150,56
True,3393,47


## 23. Registros ZM01 separados pela duração

A aba `Outliers_removidos` contém registros acima do limite de duração usado na limpeza. Uma duração elevada não é, por si só, prova de erro. Esta auditoria identifica quais registros poderiam afetar o alvo de PDM no recorte.

A extração reutiliza a regra de identificação já presente na seção 4. Os registros continuam separados: sua inclusão dependerá da revisão do local, da data e da validade da ocorrência. A duração final e o texto atualizado da ordem não são utilizados como variáveis históricas.

In [27]:
zm01_outliers = auditorias[("ZM01", "Outliers_removidos")][[
    "ordem", "nota", "local_de_instalacao", "inicio_avaria_datetime",
    "fim_avaria_datetime", "duracao_avaria_horas", "falha_flag",
]].copy()

identificacao_outliers = extrair_ativos(zm01_outliers["local_de_instalacao"])
zm01_outliers = zm01_outliers.join(identificacao_outliers[colunas_identificacao])
zm01_outliers["data_avaria"] = pd.to_datetime(
    zm01_outliers["inicio_avaria_datetime"], errors="coerce"
).dt.normalize()

outliers_pdm_recorte = zm01_outliers.loc[
    zm01_outliers["tipo_ativo"].eq(TIPO_ATIVO)
    & zm01_outliers["chave_candidata"].notna()
    & zm01_outliers["data_avaria"].between(DATA_INICIO, DATA_FIM)
].copy()

print("Registros na aba de outliers:", len(zm01_outliers))
print("Registros PDM identificados dentro do período:", len(outliers_pdm_recorte))
print("Desses, sem porta no calendário atual:",
      (~outliers_pdm_recorte["chave_candidata"].isin(portas["chave_candidata"])).sum())

display(outliers_pdm_recorte[[
    "ordem", "chave_candidata", "data_avaria", "duracao_avaria_horas", "falha_flag",
]].head(10))


Registros na aba de outliers: 103
Registros PDM identificados dentro do período: 40
Desses, sem porta no calendário atual: 0


,ordem,chave_candidata,data_avaria,duracao_avaria_horas,falha_flag
8,53465108.0,BUT|2|4|PDM,2025-02-10,1204.500000,False
27,53909221.0,FRE|2|4|PDM,2025-10-19,25.500000,False
29,53643455.0,HIG|2|14|PDM,2025-05-20,28.333333,False
31,52960553.0,LUZ|1|4|PDM,2024-05-15,2854.288889,True
32,54227610.0,LUZ|1|23|PDM,2026-03-21,69.395278,False
33,52761904.0,LUZ|2|4|PDM,2024-01-26,1454.000000,True
34,54227707.0,LUZ|2|10|PDM,2026-03-26,278.603333,False
35,54227520.0,LUZ|2|13|PDM,2026-03-13,64.232500,False
36,54227621.0,LUZ|2|17|PDM,2026-03-26,273.609167,False
37,54227445.0,LUZ|2|23|PDM,2026-03-07,206.749167,False


## 24. Comparação dos critérios por porta e dia

A comparação utiliza a base principal ZM01, com ordem identificada e sem divergência de local/dia, conforme a seção 16. Registros separados para revisão e outliers ainda não entram nos candidatos.

O agrupamento reúne registros da mesma porta no mesmo dia. Uma marcação `True` em qualquer registro torna o dia positivo no critério restrito. Isso evita escolher arbitrariamente a marcação de uma versão repetida da ordem.

A coluna `tem_zm01_flag` descreve a presença de uma marcação; seu valor zero não é um rótulo de ausência de falha. A tabela não reconstrói episódios contínuos nem prova quando os dados foram disponibilizados.

In [28]:
zm01_candidatas = corretivas_sem_ambiguidade["ZM01"].loc[
    corretivas_sem_ambiguidade["ZM01"]["chave_candidata"].isin(portas["chave_candidata"])
    & corretivas_sem_ambiguidade["ZM01"]["data_avaria"].between(DATA_INICIO, DATA_FIM)
].copy()

assert zm01_candidatas["falha_flag"].isin([True, False]).all()

eventos_zm01_candidatos = (
    zm01_candidatas.groupby(["chave_candidata", "data_avaria"])
    .agg(tem_zm01_flag=("falha_flag", "max"))
    .reset_index()
)
eventos_zm01_candidatos["tem_zm01"] = 1
eventos_zm01_candidatos["tem_zm01_flag"] = eventos_zm01_candidatos["tem_zm01_flag"].astype(int)

comparacao_criterios = pd.DataFrame({
    "criterio": ["Toda ZM01 individual elegível", "ZM01 individual com marcação de falha"],
    "dias_porta_com_evidencia": [
        len(eventos_zm01_candidatos),
        eventos_zm01_candidatos["tem_zm01_flag"].sum(),
    ],
})

display(comparacao_criterios)
display(eventos_zm01_candidatos.head())


,criterio,dias_porta_com_evidencia
0,Toda ZM01 individual elegível,6228
1,ZM01 individual com marcação de falha,1962


,chave_candidata,data_avaria,tem_zm01_flag,tem_zm01
0,BUT|1|10|PDM,2023-11-10,1,1
1,BUT|1|10|PDM,2023-11-12,1,1
2,BUT|1|10|PDM,2023-11-13,1,1
3,BUT|1|10|PDM,2023-12-09,1,1
4,BUT|1|10|PDM,2023-12-20,1,1


## 25. Presença de registros de alarmes por estação e mês

O diagnóstico conta dias com pelo menos um registro PDM na estação. Ele ajuda a localizar períodos que precisam de investigação. **Dias sem registros não comprovam falha na coleta nem ausência de alarmes.** Mesmo um dia com registros em uma porta não comprova coleta completa nas demais.

Nesta tabela de auditoria, zero significa literalmente zero registros encontrados. Esse preenchimento não altera os valores ausentes das variáveis da tabela mestra. O denominador usa apenas os dias do recorte, inclusive nos meses inicial e final.

In [29]:
alarmes_recorte_auditoria = alarmes_pdm.loc[
    alarmes_pdm["data_alarme"].between(DATA_INICIO, DATA_FIM)
    & alarmes_pdm["estacao"].isin(portas["estacao"])
].copy()
alarmes_recorte_auditoria["mes"] = alarmes_recorte_auditoria["data_alarme"].dt.to_period("M")

presenca_mensal = (
    alarmes_recorte_auditoria.groupby(["estacao", "mes"])
    .agg(
        dias_com_registro=("data_alarme", "nunique"),
        qtd_registros=("data_alarme", "size"),
    )
    .reset_index()
)

dias_por_mes = (
    calendario["data"].dt.to_period("M").value_counts()
    .sort_index().rename_axis("mes").reset_index(name="dias_no_recorte")
)
estacoes = portas[["estacao"]].drop_duplicates()
auditoria_mensal = estacoes.merge(dias_por_mes, how="cross")
auditoria_mensal = auditoria_mensal.merge(
    presenca_mensal, on=["estacao", "mes"], how="left", validate="one_to_one"
)
auditoria_mensal[["dias_com_registro", "qtd_registros"]] = (
    auditoria_mensal[["dias_com_registro", "qtd_registros"]].fillna(0).astype(int)
)
auditoria_mensal["dias_sem_registro"] = (
    auditoria_mensal["dias_no_recorte"] - auditoria_mensal["dias_com_registro"]
)

assert auditoria_mensal["dias_sem_registro"].ge(0).all()
assert auditoria_mensal["qtd_registros"].sum() == len(alarmes_recorte_auditoria)

print("Combinações estação/mês sem registros:",
      auditoria_mensal["qtd_registros"].eq(0).sum())
display(auditoria_mensal.sort_values(
    ["dias_sem_registro", "estacao", "mes"], ascending=[False, True, True]
).head(20))


Combinações estação/mês sem registros: 78


,estacao,mes,dias_no_recorte,dias_com_registro,qtd_registros,dias_sem_registro
51,FAL,2025-07,31,0,0,31
52,FAL,2025-08,31,0,0,31
54,FAL,2025-10,31,0,0,31
56,FAL,2025-12,31,0,0,31
92,FRE,2023-12,31,0,0,31
93,FRE,2024-01,31,0,0,31
95,FRE,2024-03,31,0,0,31
97,FRE,2024-05,31,0,0,31
99,FRE,2024-07,31,0,0,31
100,FRE,2024-08,31,0,0,31


## 26. Prévia dos positivos nos próximos sete dias

A prévia apresenta os dois critérios em colunas separadas, sem escolher o alvo definitivo. Uma ZM01 candidata com início em 10/01 torna positivas as datas de referência entre 03/01 e 09/01: a ocorrência está entre D+1 e D+7. O próprio dia 10/01 não fica positivo por essa ocorrência.

O código combina cada dia com evidência com as antecedências de 1 a 7 dias. Depois, reúne as datas de referência iguais. Essa abordagem evita criar uma função ou usar janelas móveis nesta etapa.

**Os únicos valores da prévia são 1 e ausente.** Um valor 1 indica evidência pelo critério candidato; ausente não significa 0. Sem confirmação de cobertura e revisão dos critérios, não há base suficiente para gerar negativos confiáveis. Os últimos sete dias do calendário também permanecem ausentes para manter somente horizontes inteiramente dentro do recorte.

In [30]:
antecedencias = pd.DataFrame({"dias_antes": range(1, 8)})
referencias_candidatas = eventos_zm01_candidatos.merge(antecedencias, how="cross")

referencias_candidatas["data"] = (
    referencias_candidatas["data_avaria"]
    - pd.to_timedelta(referencias_candidatas["dias_antes"], unit="D")
)

# Apenas datas D com os sete dias seguintes dentro do calendário.
ultimo_dia_com_horizonte = DATA_FIM - pd.Timedelta(days=7)
referencias_candidatas = referencias_candidatas.loc[
    referencias_candidatas["data"].between(DATA_INICIO, ultimo_dia_com_horizonte)
].copy()

diferenca_dias = (
    referencias_candidatas["data_avaria"] - referencias_candidatas["data"]
).dt.days
assert diferenca_dias.between(1, 7).all()

positivos_candidatos = (
    referencias_candidatas.groupby(["chave_candidata", "data"])[
        ["tem_zm01", "tem_zm01_flag"]
    ]
    .max()
    .reset_index()
    .rename(columns={
        "tem_zm01": "previa_falha_7d_toda_zm01",
        "tem_zm01_flag": "previa_falha_7d_zm01_flag",
    })
)

# Sem marcação positiva no critério restrito, o resultado permanece desconhecido.
positivos_candidatos["previa_falha_7d_zm01_flag"] = (
    positivos_candidatos["previa_falha_7d_zm01_flag"]
    .where(positivos_candidatos["previa_falha_7d_zm01_flag"].eq(1), pd.NA)
    .astype("Int64")
)

tabela_mestra_previa_alvo = tabela_mestra_manutencoes.merge(
    positivos_candidatos,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)
colunas_previa = ["previa_falha_7d_toda_zm01", "previa_falha_7d_zm01_flag"]
tabela_mestra_previa_alvo[colunas_previa] = tabela_mestra_previa_alvo[colunas_previa].astype("Int64")

assert len(tabela_mestra_previa_alvo) == len(tabela_mestra_manutencoes)
assert not tabela_mestra_previa_alvo.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_previa_alvo[colunas_previa].stack().eq(1).all()
assert tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["data"] > ultimo_dia_com_horizonte, colunas_previa
].isna().all().all()
assert tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["previa_falha_7d_zm01_flag"].eq(1),
    "previa_falha_7d_toda_zm01",
].eq(1).all()

display(tabela_mestra_previa_alvo[colunas_previa].notna().sum().to_frame("positivos_candidatos"))
display(tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["previa_falha_7d_toda_zm01"].eq(1),
    ["chave_candidata", "data", *colunas_previa],
].head(10))


,positivos_candidatos
previa_falha_7d_toda_zm01,36715
previa_falha_7d_zm01_flag,12539


,chave_candidata,data,previa_falha_7d_toda_zm01,previa_falha_7d_zm01_flag
17,BUT|1|10|PDM,2023-11-03,1,1
18,BUT|1|10|PDM,2023-11-04,1,1
19,BUT|1|10|PDM,2023-11-05,1,1
20,BUT|1|10|PDM,2023-11-06,1,1
21,BUT|1|10|PDM,2023-11-07,1,1
22,BUT|1|10|PDM,2023-11-08,1,1
23,BUT|1|10|PDM,2023-11-09,1,1
24,BUT|1|10|PDM,2023-11-10,1,1
25,BUT|1|10|PDM,2023-11-11,1,1
26,BUT|1|10|PDM,2023-11-12,1,1


### Resultado da revisão do alvo

A seção apresenta a comparação dos critérios ZM01, os outliers PDM candidatos à revisão, o diagnóstico mensal dos alarmes e uma prévia dos positivos de sete dias. A variável `tabela_mestra_previa_alvo` mantém as duas alternativas separadas.

A prévia **não é uma base de treinamento**: não contém negativos definidos. Valores ausentes não devem ser preenchidos com zero nesta etapa. Nenhuma correlação entre alarmes e manutenção foi presumida.

A definição do alvo final depende de três decisões: o critério de falha, o tratamento dos registros separados e o período em que o registro de falhas pode ser considerado completo. A cobertura dos alarmes precisa de avaliação própria para as entradas temporais. Até essas decisões, a prévia permanece exploratória e nenhuma base é exportada.

## 27. Critério inicial do alvo

A primeira versão adota **toda ZM01 individual elegível como indicador de falha**, independentemente de `falha_flag`. Trata-se de uma definição operacional inicial: uma ordem emergencial serve como evidência, mas não comprova sozinha uma quebra física.

A elegibilidade segue as seções anteriores: PDM identificada, ordem presente, ausência de divergência de local/dia e início de avaria dentro do recorte. Os registros separados na auditoria ainda não entram nesse critério. ZM02 e ZM03 permanecem como fontes de histórico ou contexto, sem gerar positivos automaticamente.

A coluna `falha_7d` recebe a prévia correspondente a esse critério. Ela continua **parcial**: contém 1 quando há evidência elegível nos sete dias seguintes e ausente nos demais casos. A geração de zero depende de cobertura confirmada e da revisão dos registros separados.

As colunas de alvo e status descrevem o resultado futuro. Elas nunca devem integrar as entradas da LSTM.

In [31]:
tabela_mestra_historico = (
    tabela_mestra_previa_alvo
    .sort_values(["chave_candidata", "data"])
    .reset_index(drop=True)
    .copy()
)

tabela_mestra_historico["falha_7d"] = (
    tabela_mestra_historico["previa_falha_7d_toda_zm01"].astype("Int64")
)

tabela_mestra_historico["status_rotulo"] = "sem negativo definido"
tabela_mestra_historico.loc[
    tabela_mestra_historico["falha_7d"].eq(1),
    "status_rotulo",
] = "positivo pelo critério ZM01"

tabela_mestra_historico.loc[
    tabela_mestra_historico["data"] > ultimo_dia_com_horizonte,
    "status_rotulo",
] = "horizonte ultrapassa o recorte"

display(tabela_mestra_historico["status_rotulo"].value_counts().to_frame("linhas"))


,linhas
status_rotulo,
sem negativo definido,431621
positivo pelo critério ZM01,36715
horizonte ultrapassa o recorte,3696


## 28. Significado de dias desde a última ZM01/ZM02

As duas primeiras variáveis históricas medem os dias desde o último **início de avaria registrado** em uma ordem elegível da mesma porta. Elas não medem tempo desde uma manutenção executada.

O cálculo considera uma previsão feita ao final de D. Por isso, uma avaria registrada em D produz valor zero nesse dia. Sem um novo registro, a diferença aumenta com a passagem dos dias:

| Data de referência | Último início de avaria registrado | Dias desde o registro |
|---|---|---:|
| 10/01 | 10/01 | 0 |
| 11/01 | 10/01 | 1 |
| 12/01 | 10/01 | 2 |

Sem histórico anterior elegível, o resultado permanece ausente. O valor zero não substitui histórico desconhecido.

O cálculo também aproveita registros anteriores a `DATA_INICIO`, evitando reiniciar o histórico artificialmente no primeiro dia do calendário. A sequência fica ordenada por porta e data.

Esta etapa usa as datas de ocorrência existentes na extração. Ainda será necessário confirmar quando cada registro ficou disponível: um início de avaria preenchido posteriormente não poderia ter sido usado numa previsão passada.

## 29. Dias desde a última ZM01 registrada

O código identifica a última avaria elegível anterior ao calendário, marca os dias que têm registros no recorte e propaga a última data apenas para a frente, dentro de cada porta.

- `groupby().max()` obtém a última data anterior ao recorte para cada porta.
- `where()` conserva a data somente nas linhas com um registro ZM01.
- `groupby().ffill()` propaga a última data conhecida dentro da mesma porta.
- `map()` recupera o histórico anterior quando ainda não houve registro no calendário.

Esse preenchimento aplica-se à **data da última ocorrência registrada**. Ele não preenche contagens ausentes de alarmes ou rótulos negativos.

In [32]:
# Histórico elegível anterior ao início do calendário.
historico_zm01 = corretivas_sem_ambiguidade["ZM01"]
zm01_anteriores = historico_zm01.loc[
    historico_zm01["chave_candidata"].isin(portas["chave_candidata"])
    & historico_zm01["data_avaria"].lt(DATA_INICIO)
]
ultima_zm01_antes = zm01_anteriores.groupby("chave_candidata")["data_avaria"].max()

# Marcação das datas com registros dentro do calendário.
tem_zm01_no_dia = tabela_mestra_historico["qtd_ordens_zm01_inicio_avaria"].gt(0)
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico["data"].where(tem_zm01_no_dia)
)

# Propagação para a frente, sempre dentro da mesma porta.
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico.groupby("chave_candidata")["data_ultima_zm01_registrada"].ffill()
)

# Uso da última avaria anterior ao calendário quando houver registro.
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico["data_ultima_zm01_registrada"].fillna(
        tabela_mestra_historico["chave_candidata"].map(ultima_zm01_antes)
    )
)

tabela_mestra_historico["dias_desde_ultima_zm01"] = (
    tabela_mestra_historico["data"]
    - tabela_mestra_historico["data_ultima_zm01_registrada"]
).dt.days.astype("Int64")

display(tabela_mestra_historico[[
    "chave_candidata", "data", "data_ultima_zm01_registrada", "dias_desde_ultima_zm01",
]].head(10))


,chave_candidata,data,data_ultima_zm01_registrada,dias_desde_ultima_zm01
0,BUT|1|10|PDM,2023-10-17,2023-07-16,93
1,BUT|1|10|PDM,2023-10-18,2023-07-16,94
2,BUT|1|10|PDM,2023-10-19,2023-07-16,95
3,BUT|1|10|PDM,2023-10-20,2023-07-16,96
4,BUT|1|10|PDM,2023-10-21,2023-07-16,97
5,BUT|1|10|PDM,2023-10-22,2023-07-16,98
6,BUT|1|10|PDM,2023-10-23,2023-07-16,99
7,BUT|1|10|PDM,2023-10-24,2023-07-16,100
8,BUT|1|10|PDM,2023-10-25,2023-07-16,101
9,BUT|1|10|PDM,2023-10-26,2023-07-16,102


## 30. Dias desde a última ZM02 registrada

O cálculo repete os passos da ZM01 para facilitar a comparação entre as duas fontes. O nome `dias_desde_ultima_zm02` mantém a nomenclatura proposta para o modelo, mas sua referência atual é o início da avaria da ZM02, e não a data programada ou executada da manutenção.

A variável `dias_desde_ultima_zm03` ainda não é criada: as datas disponíveis de ZM03 foram tratadas como programação, e não há identificação individual de porta ou confirmação da execução.

In [33]:
historico_zm02 = corretivas_sem_ambiguidade["ZM02"]
zm02_anteriores = historico_zm02.loc[
    historico_zm02["chave_candidata"].isin(portas["chave_candidata"])
    & historico_zm02["data_avaria"].lt(DATA_INICIO)
]
ultima_zm02_antes = zm02_anteriores.groupby("chave_candidata")["data_avaria"].max()

tem_zm02_no_dia = tabela_mestra_historico["qtd_ordens_zm02_inicio_avaria"].gt(0)
tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico["data"].where(tem_zm02_no_dia)
)

tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico.groupby("chave_candidata")["data_ultima_zm02_registrada"].ffill()
)

tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico["data_ultima_zm02_registrada"].fillna(
        tabela_mestra_historico["chave_candidata"].map(ultima_zm02_antes)
    )
)

tabela_mestra_historico["dias_desde_ultima_zm02"] = (
    tabela_mestra_historico["data"]
    - tabela_mestra_historico["data_ultima_zm02_registrada"]
).dt.days.astype("Int64")

display(tabela_mestra_historico[[
    "chave_candidata", "data", "data_ultima_zm02_registrada", "dias_desde_ultima_zm02",
]].head(10))


,chave_candidata,data,data_ultima_zm02_registrada,dias_desde_ultima_zm02
0,BUT|1|10|PDM,2023-10-17,NaT,<NA>
1,BUT|1|10|PDM,2023-10-18,NaT,<NA>
2,BUT|1|10|PDM,2023-10-19,NaT,<NA>
3,BUT|1|10|PDM,2023-10-20,NaT,<NA>
4,BUT|1|10|PDM,2023-10-21,NaT,<NA>
5,BUT|1|10|PDM,2023-10-22,NaT,<NA>
6,BUT|1|10|PDM,2023-10-23,NaT,<NA>
7,BUT|1|10|PDM,2023-10-24,NaT,<NA>
8,BUT|1|10|PDM,2023-10-25,NaT,<NA>
9,BUT|1|10|PDM,2023-10-26,NaT,<NA>


## 31. Conferência das variáveis históricas

As verificações conferem que as datas utilizadas não ultrapassam D, que os dias desde a última ocorrência nunca são negativos e que um registro no próprio dia resulta em zero. Uma comparação no início do calendário confere o aproveitamento do histórico anterior.

Essas verificações validam o cálculo pelas datas de ocorrência. Elas não substituem a confirmação de disponibilidade histórica dos dados.

In [34]:
assert len(tabela_mestra_historico) == len(tabela_mestra_manutencoes)
assert not tabela_mestra_historico.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_historico["falha_7d"].dropna().eq(1).all()

for tipo in ["zm01", "zm02"]:
    coluna_data = f"data_ultima_{tipo}_registrada"
    coluna_dias = f"dias_desde_ultima_{tipo}"
    coluna_contagem = f"qtd_ordens_{tipo}_inicio_avaria"

    tem_historico = tabela_mestra_historico[coluna_data].notna()
    tem_registro_hoje = tabela_mestra_historico[coluna_contagem].gt(0)

    assert (
        tabela_mestra_historico.loc[tem_historico, coluna_data]
        <= tabela_mestra_historico.loc[tem_historico, "data"]
    ).all()
    assert tabela_mestra_historico[coluna_dias].dropna().ge(0).all()
    assert tabela_mestra_historico.loc[tem_registro_hoje, coluna_dias].eq(0).all()

inicio_calendario = tabela_mestra_historico.loc[
    tabela_mestra_historico["data"].eq(DATA_INICIO)
].copy()
sem_zm01_no_inicio = inicio_calendario["qtd_ordens_zm01_inicio_avaria"].isna()
esperado_zm01 = inicio_calendario.loc[
    sem_zm01_no_inicio, "chave_candidata"
].map(ultima_zm01_antes)
pd.testing.assert_series_equal(
    inicio_calendario.loc[sem_zm01_no_inicio, "data_ultima_zm01_registrada"],
    esperado_zm01,
    check_names=False,
)

sem_zm02_no_inicio = inicio_calendario["qtd_ordens_zm02_inicio_avaria"].isna()
esperado_zm02 = inicio_calendario.loc[
    sem_zm02_no_inicio, "chave_candidata"
].map(ultima_zm02_antes)
pd.testing.assert_series_equal(
    inicio_calendario.loc[sem_zm02_no_inicio, "data_ultima_zm02_registrada"],
    esperado_zm02,
    check_names=False,
)

colunas_historico = ["dias_desde_ultima_zm01", "dias_desde_ultima_zm02"]
display(tabela_mestra_historico[colunas_historico].describe())
display(tabela_mestra_historico[colunas_historico].isna().sum().to_frame("linhas_sem_historico"))

print("Linhas da tabela:", len(tabela_mestra_historico))
print("Positivos pelo critério ZM01:", tabela_mestra_historico["falha_7d"].eq(1).sum())
print("Negativos definidos:", tabela_mestra_historico["falha_7d"].eq(0).sum())


,dias_desde_ultima_zm01,dias_desde_ultima_zm02
count,449751.0,228377.0
mean,109.416691,319.799796
std,128.96424,249.255303
min,0.0,0.0
25%,25.0,115.0
50%,66.0,266.0
75%,145.0,471.0
max,1092.0,1180.0


,linhas_sem_historico
dias_desde_ultima_zm01,22281
dias_desde_ultima_zm02,243655


Linhas da tabela: 472032
Positivos pelo critério ZM01: 36715
Negativos definidos: 0


### Resultado das primeiras variáveis históricas

A variável `tabela_mestra_historico` contém o alvo parcial pelo critério ZM01 e as variáveis `dias_desde_ultima_zm01` e `dias_desde_ultima_zm02`. As duas colunas auxiliares com a data do último registro permitem conferir o cálculo; não são novas variáveis numéricas para a LSTM.

O resultado continua exploratório. A base ainda não contém negativos definidos, a cobertura permanece pendente e os registros separados para revisão não foram incorporados. A coluna `status_rotulo` explicita essas limitações por linha.

O próximo cálculo será o histórico de alarmes em janelas de 7 e 14 dias. A definição de ausência de registros terá de permanecer explícita para não transformar falta de informação em zero eventos. A montagem das sequências e o treinamento dependem da resolução dessas questões.

## 32. Definição da unidade de contagem dos alarmes

A base contém linhas com a mesma TAG e o mesmo início de alarme, mas diferentes horários de normalização. A nova contagem define uma **ativação registrada** como um par único de TAG e início do alarme. A definição é operacional e poderá ser revista se houver um identificador de evento mais preciso.

O código seleciona somente identificação e início. Assim, `drop_duplicates()` não escolhe qual horário de normalização está correto. A análise da duração permanece pendente.

A coluna anterior `qtd_registros_alarmes` continua como auditoria das linhas da fonte. As novas colunas usam o nome `qtd_alarmes_registrados` e contam ativações distintas. A mudança de unidade de contagem não altera as tabelas anteriores.

In [35]:
colunas_ativacao = [
    "chave_candidata", "tag", "inicio_alarme", "data_alarme",
]

# Seleção das portas do calendário, preservando o histórico anterior ao recorte.
registros_portas = alarmes_identificados.loc[
    alarmes_identificados["chave_candidata"].isin(portas["chave_candidata"]),
    colunas_ativacao,
].copy()

# A mesma TAG e o mesmo instante de início devem identificar o mesmo ativo.
identificacoes_por_inicio = (
    registros_portas.groupby(["tag", "inicio_alarme"])["chave_candidata"].nunique()
)
assert identificacoes_por_inicio.le(1).all()

ativacoes_alarmes = registros_portas.drop_duplicates(["tag", "inicio_alarme"]).copy()

ativacoes_diarias = (
    ativacoes_alarmes.groupby(["chave_candidata", "data_alarme"])
    .size()
    .reset_index(name="qtd_alarmes_registrados_1d")
    .rename(columns={"data_alarme": "data"})
)

assert ativacoes_diarias["qtd_alarmes_registrados_1d"].sum() == len(ativacoes_alarmes)

print("Linhas de alarmes das portas selecionadas:", len(registros_portas))
print("Ativações distintas registradas:", len(ativacoes_alarmes))
print("Linhas excedentes na contagem de ativações:", len(registros_portas) - len(ativacoes_alarmes))
display(ativacoes_diarias.head())


Linhas de alarmes das portas selecionadas: 678166
Ativações distintas registradas: 304143
Linhas excedentes na contagem de ativações: 374023


,chave_candidata,data,qtd_alarmes_registrados_1d
0,BUT|1|10|PDM,2023-11-10,3
1,BUT|1|10|PDM,2023-11-11,3
2,BUT|1|10|PDM,2023-11-12,1
3,BUT|1|10|PDM,2023-11-13,10
4,BUT|1|10|PDM,2023-11-21,2


## 33. Calendário auxiliar com 13 dias anteriores

Uma janela de 14 dias em D usa D−13 até D. Por isso, o calendário auxiliar começa 13 dias antes de `DATA_INICIO`. A extensão permite calcular a primeira janela inteira do calendário principal, sem usar dias futuros.

Nesta tabela auxiliar, **zero significa que nenhuma ativação foi encontrada no arquivo para aquela porta e dia**. Isso não comprova ausência de alarmes na operação. As colunas de auditoria anteriores continuam com seus valores ausentes, e o alvo não recebe zeros.

A coluna `dia_com_registro` marca dias em que há pelo menos uma ativação. Ela indica presença de registros, não cobertura completa da coleta.

In [36]:
inicio_historico_alarmes = DATA_INICIO - pd.Timedelta(days=13)

calendario_alarmes = pd.DataFrame({
    "data": pd.date_range(inicio_historico_alarmes, DATA_FIM, freq="D")
})
historico_alarmes_diario = portas[["chave_candidata"]].merge(
    calendario_alarmes, how="cross"
)

historico_alarmes_diario = historico_alarmes_diario.merge(
    ativacoes_diarias,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

historico_alarmes_diario["dia_com_registro"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_1d"].notna().astype(int)
)

# Zero apenas na contagem de ativações encontradas no arquivo.
historico_alarmes_diario["qtd_alarmes_registrados_1d"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_1d"].fillna(0).astype(int)
)

historico_alarmes_diario = (
    historico_alarmes_diario
    .sort_values(["chave_candidata", "data"])
    .reset_index(drop=True)
)

assert len(historico_alarmes_diario) == len(portas) * len(calendario_alarmes)
assert not historico_alarmes_diario.duplicated(["chave_candidata", "data"]).any()
display(historico_alarmes_diario.head())


,chave_candidata,data,qtd_alarmes_registrados_1d,dia_com_registro
0,BUT|1|10|PDM,2023-10-04,0,0
1,BUT|1|10|PDM,2023-10-05,0,0
2,BUT|1|10|PDM,2023-10-06,0,0
3,BUT|1|10|PDM,2023-10-07,0,0
4,BUT|1|10|PDM,2023-10-08,0,0


## 34. Contagens móveis de 7 e 14 dias

`rolling(7)` percorre grupos de sete linhas consecutivas. Como o calendário tem exatamente uma linha por porta e dia, sete linhas representam sete dias. O agrupamento por chave impede a mistura entre portas.

- Janela de 7 dias: D−6 até D.
- Janela de 14 dias: D−13 até D.
- `min_periods` exige que a janela tenha a quantidade completa de dias do calendário auxiliar.

A saída do agrupamento tem um índice adicional com a chave da porta. `reset_index(level=0, drop=True)` remove esse nível e permite alinhar os resultados às linhas originais.

A previsão é feita ao final de D, portanto os registros de D entram na janela. Os nomes deixam explícito que o cálculo representa alarmes **registrados**, cuja completude ainda não foi confirmada.

In [37]:
for dias in [7, 14]:
    historico_alarmes_diario[f"qtd_alarmes_registrados_{dias}d"] = (
        historico_alarmes_diario.groupby("chave_candidata")["qtd_alarmes_registrados_1d"]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

    historico_alarmes_diario[f"dias_com_registro_{dias}d"] = (
        historico_alarmes_diario.groupby("chave_candidata")["dia_com_registro"]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

display(historico_alarmes_diario.loc[
    historico_alarmes_diario["data"].eq(DATA_INICIO)
].head())


,chave_candidata,data,qtd_alarmes_registrados_1d,dia_com_registro,qtd_alarmes_registrados_7d,dias_com_registro_7d,qtd_alarmes_registrados_14d,dias_com_registro_14d
13,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0
920,BUT|1|11|PDM,2023-10-17,0,0,0,0,0,0
1827,BUT|1|12|PDM,2023-10-17,0,0,0,0,0,0
2734,BUT|1|13|PDM,2023-10-17,0,0,0,0,0,0
3641,BUT|1|14|PDM,2023-10-17,0,0,0,0,0,0


## 35. Comparação entre duas semanas consecutivas

A diferença entre a soma de 14 dias e a soma de 7 dias fornece a semana anterior: D−13 até D−7. A variação compara essa semana com D−6 até D.

Por exemplo, 12 ativações na semana atual e 8 na anterior produzem variação +4. Uma variação negativa indica redução de registros. O cálculo usa diferença absoluta, sem divisão.

A coluna `variacao_alarmes_registrados_7d` é a definição inicial proposta para a ideia de aceleração de alarmes. Ela pode mudar tanto por alteração nas ocorrências quanto por alteração na coleta. As colunas de dias com registro ajudam na investigação, mas não corrigem uma coleta incompleta.

In [38]:
historico_alarmes_diario["qtd_alarmes_registrados_7d_anteriores"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_14d"]
    - historico_alarmes_diario["qtd_alarmes_registrados_7d"]
)

historico_alarmes_diario["variacao_alarmes_registrados_7d"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_7d"]
    - historico_alarmes_diario["qtd_alarmes_registrados_7d_anteriores"]
)

colunas_janelas = [
    "qtd_alarmes_registrados_1d",
    "qtd_alarmes_registrados_7d",
    "qtd_alarmes_registrados_14d",
    "qtd_alarmes_registrados_7d_anteriores",
    "variacao_alarmes_registrados_7d",
    "dias_com_registro_7d",
    "dias_com_registro_14d",
]

janelas_alarmes = historico_alarmes_diario.loc[
    historico_alarmes_diario["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_janelas],
].copy()

assert janelas_alarmes[colunas_janelas].notna().all().all()
display(janelas_alarmes.head(10))


,chave_candidata,data,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
13,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0,0
14,BUT|1|10|PDM,2023-10-18,0,0,0,0,0,0,0
15,BUT|1|10|PDM,2023-10-19,0,0,0,0,0,0,0
16,BUT|1|10|PDM,2023-10-20,0,0,0,0,0,0,0
17,BUT|1|10|PDM,2023-10-21,0,0,0,0,0,0,0
18,BUT|1|10|PDM,2023-10-22,0,0,0,0,0,0,0
19,BUT|1|10|PDM,2023-10-23,0,0,0,0,0,0,0
20,BUT|1|10|PDM,2023-10-24,0,0,0,0,0,0,0
21,BUT|1|10|PDM,2023-10-25,0,0,0,0,0,0,0
22,BUT|1|10|PDM,2023-10-26,0,0,0,0,0,0,0


## 36. Inclusão das janelas na tabela mestra

A junção acrescenta as sete colunas à tabela histórica, por chave e data. O calendário principal permanece no mesmo período. Os 13 dias auxiliares servem somente ao cálculo das janelas e não acrescentam linhas à tabela final.

In [39]:
tabela_mestra_janelas = tabela_mestra_historico.merge(
    janelas_alarmes,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_janelas) == len(tabela_mestra_historico)
assert not tabela_mestra_janelas.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_janelas["falha_7d"].equals(tabela_mestra_historico["falha_7d"])
assert tabela_mestra_janelas["qtd_registros_alarmes"].equals(
    tabela_mestra_historico["qtd_registros_alarmes"]
)
assert tabela_mestra_janelas[colunas_janelas].notna().all().all()

print("Linhas da tabela:", len(tabela_mestra_janelas))
print("Novas colunas:", len(colunas_janelas))
display(tabela_mestra_janelas[colunas_janelas].describe())


Linhas da tabela: 472032
Novas colunas: 7


,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
count,472032.000000,472032.0,472032.0,472032.0,472032.0,472032.0,472032.0
mean,0.641747,4.489842,8.970616,4.480775,0.009067,0.703266,1.401966
std,4.575306,21.599751,37.197844,21.599121,21.965039,1.320716,2.406354
min,0.000000,0.0,0.0,0.0,-1854.0,0.0,0.0
25%,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
50%,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
75%,0.000000,4.0,8.0,4.0,0.0,1.0,2.0
max,375.000000,1956.0,2501.0,1956.0,1744.0,7.0,14.0


## 37. Conferência das janelas por contagem direta

A conferência seleciona o início e o fim do calendário e uma linha com a maior contagem diária. Para cada caso, o código conta diretamente as ativações da mesma porta dentro do intervalo de datas, sem usar `rolling()`.

A soma de 14 dias deve ser igual à soma da semana atual com a semana anterior. Os dias com registros devem permanecer entre zero e o tamanho da janela.

In [40]:
assert (
    tabela_mestra_janelas["qtd_alarmes_registrados_14d"]
    == tabela_mestra_janelas["qtd_alarmes_registrados_7d"]
    + tabela_mestra_janelas["qtd_alarmes_registrados_7d_anteriores"]
).all()

assert tabela_mestra_janelas["dias_com_registro_7d"].between(0, 7).all()
assert tabela_mestra_janelas["dias_com_registro_14d"].between(0, 14).all()
assert tabela_mestra_janelas["qtd_alarmes_registrados_7d_anteriores"].ge(0).all()

indices_exemplo = [
    tabela_mestra_janelas.index[0],
    tabela_mestra_janelas.index[-1],
    tabela_mestra_janelas["qtd_alarmes_registrados_1d"].idxmax(),
]
amostras_janelas = tabela_mestra_janelas.loc[indices_exemplo].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_janelas.iterrows():
    registros_da_porta = ativacoes_alarmes.loc[
        ativacoes_alarmes["chave_candidata"].eq(linha["chave_candidata"])
    ]

    for dias in [1, 7, 14]:
        inicio_janela = linha["data"] - pd.Timedelta(days=dias - 1)
        quantidade_direta = registros_da_porta["data_alarme"].between(
            inicio_janela, linha["data"]
        ).sum()
        assert quantidade_direta == linha[f"qtd_alarmes_registrados_{dias}d"]

display(amostras_janelas[["chave_candidata", "data", *colunas_janelas]])
print("Contagens móveis conferidas com as ativações de origem.")


,chave_candidata,data,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
0,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0,0
472031,VSO|2|9|PDM,2026-03-28,0,0,0,0,0,0,0
208046,HIG|2|2|PDM,2025-07-16,375,388,388,0,388,2,2


Contagens móveis conferidas com as ativações de origem.


### Resultado das janelas de alarmes

A variável `tabela_mestra_janelas` acrescenta contagens de ativações registradas em 1, 7 e 14 dias, a semana anterior, a variação semanal e os dias com registros. As contagens de linhas da fonte e os alvos parciais anteriores permanecem intactos.

A unidade de contagem agora é TAG + início do alarme. A ambiguidade dos horários de normalização ainda precisa de tratamento antes das variáveis de duração e anomalias de 15/30 segundos.

As novas contagens são candidatas a entradas temporais. Zero significa nenhuma ativação encontrada no arquivo; a interpretação operacional exige confirmação de cobertura. `dias_com_registro` não é uma certificação de coleta completa. O alvo continua sem negativos definidos, portanto o treinamento ainda não pode começar.

A próxima etapa poderá separar as contagens por tipo de alarme, como FO e LSD, mantendo a mesma unidade de ativação e as mesmas regras temporais.

## 38. Classificação das ativações por tipo de evento

Esta etapa prepara as variáveis `qtd_fo_7d` e `qtd_lsd_7d`, contando ativações registradas dos códigos FO e LSD. A unidade continua sendo **TAG + início do alarme**.

Parte dos registros de MBI está sem classificação preenchida no arquivo processado. Nesses casos, o código do evento pode ser recuperado da TAG. Por exemplo, `PSD_P_MBI1_PDM10FO` termina com FO. O padrão também aceita TAGs com separador, como `PSD_BUT_BUT1_PDM10_FO`.

A classificação preenchida é comparada à TAG quando ambas estão disponíveis. Uma divergência interrompe a execução para revisão. Os códigos representam tipos de evento, sem classificar automaticamente FO ou LSD como falha ou alarme grave.

In [41]:
mapa_tipos = alarmes_identificados.loc[
    alarmes_identificados["tag"].isin(ativacoes_alarmes["tag"]),
    ["tag", "tipo_evento"],
].drop_duplicates().copy()

mapa_tipos = mapa_tipos.rename(columns={"tipo_evento": "tipo_evento_processado"})
mapa_tipos["tipo_evento_processado"] = (
    mapa_tipos["tipo_evento_processado"].astype("string").str.strip().str.upper()
    .replace("", pd.NA)
)

# A extração lê somente o código após o número de PDM ou MDU.
mapa_tipos["tipo_evento_tag"] = (
    mapa_tipos["tag"].astype("string").str.strip().str.upper()
    .str.extract(r"_(?:PDM|MDU)\d+_?([A-Z][A-Z0-9]*)$", expand=False)
)

tem_duas_classificacoes = (
    mapa_tipos["tipo_evento_processado"].notna()
    & mapa_tipos["tipo_evento_tag"].notna()
)
assert (
    mapa_tipos.loc[tem_duas_classificacoes, "tipo_evento_processado"]
    == mapa_tipos.loc[tem_duas_classificacoes, "tipo_evento_tag"]
).all(), "Há divergência entre a classificação processada e a TAG."

mapa_tipos["tipo_evento"] = mapa_tipos["tipo_evento_processado"].fillna(
    mapa_tipos["tipo_evento_tag"]
)
mapa_tipos["classificacao_recuperada"] = (
    mapa_tipos["tipo_evento_processado"].isna() & mapa_tipos["tipo_evento"].notna()
)
assert not mapa_tipos["tag"].duplicated().any(), "Há mais de uma classificação por TAG."

ativacoes_classificadas = ativacoes_alarmes.merge(
    mapa_tipos[["tag", "tipo_evento", "classificacao_recuperada"]],
    on="tag",
    how="left",
    validate="many_to_one",
)
assert len(ativacoes_classificadas) == len(ativacoes_alarmes)
assert not ativacoes_classificadas.duplicated(["tag", "inicio_alarme"]).any()
assert ativacoes_classificadas["tipo_evento"].notna().all(), "Há ativações sem classificação."

print("Ativações com classificação recuperada da TAG:",
      ativacoes_classificadas["classificacao_recuperada"].sum())
display(ativacoes_classificadas["tipo_evento"].value_counts().to_frame("ativacoes"))
display(mapa_tipos.loc[
    mapa_tipos["classificacao_recuperada"],
    ["tag", "tipo_evento_processado", "tipo_evento"],
].head())


Ativações com classificação recuperada da TAG: 29583


,ativacoes
tipo_evento,
LSD,65135
OBS,56797
FC,55842
FO,55241
OA,38296
CA,16252
CSD,13314
RCE,3266


,tag,tipo_evento_processado,tipo_evento
241947,PSD_P_MBI1_PDM10FC,<NA>,FC
241949,PSD_P_MBI1_PDM10RCE,<NA>,RCE
241951,PSD_P_MBI1_PDM11FC,<NA>,FC
241954,PSD_P_MBI1_PDM11RCE,<NA>,RCE
241957,PSD_P_MBI1_PDM12CA,<NA>,CA


## 39. Contagens diárias de FO e LSD

Cada ativação recebe dois indicadores: `eh_fo` e `eh_lsd`. A soma desses indicadores fornece a quantidade diária de cada tipo. Uma ativação FO não é contada como LSD, e os demais tipos não entram nessas duas contagens.

A tabela auxiliar utiliza o mesmo calendário estendido da seção 33. Nela, zero significa nenhuma ativação **desse tipo encontrada no arquivo**. Não significa confirmação de ausência operacional do evento.

In [42]:
ativacoes_classificadas["eh_fo"] = ativacoes_classificadas["tipo_evento"].eq("FO").astype(int)
ativacoes_classificadas["eh_lsd"] = ativacoes_classificadas["tipo_evento"].eq("LSD").astype(int)

tipos_diarios = (
    ativacoes_classificadas.groupby(["chave_candidata", "data_alarme"])
    .agg(
        qtd_fo_1d=("eh_fo", "sum"),
        qtd_lsd_1d=("eh_lsd", "sum"),
    )
    .reset_index()
    .rename(columns={"data_alarme": "data"})
)

historico_tipos = historico_alarmes_diario[["chave_candidata", "data"]].merge(
    tipos_diarios,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)
historico_tipos[["qtd_fo_1d", "qtd_lsd_1d"]] = (
    historico_tipos[["qtd_fo_1d", "qtd_lsd_1d"]].fillna(0).astype(int)
)
historico_tipos = (
    historico_tipos.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
)

dentro_do_historico = ativacoes_classificadas["data_alarme"].between(
    inicio_historico_alarmes, DATA_FIM
)
assert historico_tipos["qtd_fo_1d"].sum() == ativacoes_classificadas.loc[dentro_do_historico, "eh_fo"].sum()
assert historico_tipos["qtd_lsd_1d"].sum() == ativacoes_classificadas.loc[dentro_do_historico, "eh_lsd"].sum()

display(historico_tipos.loc[
    (historico_tipos["qtd_fo_1d"] > 0) | (historico_tipos["qtd_lsd_1d"] > 0)
].head(10))


,chave_candidata,data,qtd_fo_1d,qtd_lsd_1d
37,BUT|1|10|PDM,2023-11-10,0,2
38,BUT|1|10|PDM,2023-11-11,0,3
39,BUT|1|10|PDM,2023-11-12,1,0
40,BUT|1|10|PDM,2023-11-13,1,9
48,BUT|1|10|PDM,2023-11-21,0,2
51,BUT|1|10|PDM,2023-11-24,0,1
54,BUT|1|10|PDM,2023-11-27,2,0
65,BUT|1|10|PDM,2023-12-08,0,1
72,BUT|1|10|PDM,2023-12-15,1,1
88,BUT|1|10|PDM,2023-12-31,5,0


## 40. Janelas de sete dias por tipo

O cálculo utiliza os sete dias de D−6 até D, incluindo D porque a previsão será feita ao final do dia. O agrupamento mantém cada porta separada.

Os nomes `qtd_fo_7d` e `qtd_lsd_7d` seguem a lista inicial de variáveis do projeto. Seu significado nesta versão é **quantidade de ativações registradas** de cada código nos sete dias. A interpretação como total operacional depende da confirmação de cobertura.

In [43]:
for tipo in ["fo", "lsd"]:
    historico_tipos[f"qtd_{tipo}_7d"] = (
        historico_tipos.groupby("chave_candidata")[f"qtd_{tipo}_1d"]
        .rolling(window=7, min_periods=7)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

colunas_tipos = ["qtd_fo_7d", "qtd_lsd_7d"]
janelas_tipos = historico_tipos.loc[
    historico_tipos["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_tipos],
].copy()

assert janelas_tipos[colunas_tipos].notna().all().all()
assert janelas_tipos[colunas_tipos].ge(0).all().all()

tabela_mestra_tipos = tabela_mestra_janelas.merge(
    janelas_tipos,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_tipos) == len(tabela_mestra_janelas)
assert not tabela_mestra_tipos.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_tipos["falha_7d"].equals(tabela_mestra_janelas["falha_7d"])
assert tabela_mestra_tipos["qtd_alarmes_registrados_7d"].equals(
    tabela_mestra_janelas["qtd_alarmes_registrados_7d"]
)

print("Linhas da tabela:", len(tabela_mestra_tipos))
display(tabela_mestra_tipos[colunas_tipos].describe())


Linhas da tabela: 472032


,qtd_fo_7d,qtd_lsd_7d
count,472032.0,472032.0
mean,0.818565,0.964901
std,11.637091,14.830666
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,988.0,1936.0


## 41. Conferência das contagens por tipo

FO e LSD são categorias distintas. A soma de suas contagens não pode ultrapassar a quantidade total de ativações registradas na mesma janela.

A conferência direta seleciona o início e o fim do calendário e as linhas de maior contagem FO e LSD. Para cada linha, a verificação conta as ativações da mesma porta entre D−6 e D, sem utilizar `rolling()`. Isso permite conferir a classificação, as datas e a separação entre portas.

In [44]:
assert (
    tabela_mestra_tipos["qtd_fo_7d"] + tabela_mestra_tipos["qtd_lsd_7d"]
    <= tabela_mestra_tipos["qtd_alarmes_registrados_7d"]
).all()

indices_tipos = [
    tabela_mestra_tipos.index[0],
    tabela_mestra_tipos.index[-1],
    tabela_mestra_tipos["qtd_fo_7d"].idxmax(),
    tabela_mestra_tipos["qtd_lsd_7d"].idxmax(),
]
amostras_tipos = tabela_mestra_tipos.loc[indices_tipos].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_tipos.iterrows():
    inicio_janela = linha["data"] - pd.Timedelta(days=6)
    ativacoes_da_janela = ativacoes_classificadas.loc[
        ativacoes_classificadas["chave_candidata"].eq(linha["chave_candidata"])
        & ativacoes_classificadas["data_alarme"].between(inicio_janela, linha["data"])
    ]

    for codigo in ["FO", "LSD"]:
        contagem_direta = ativacoes_da_janela["tipo_evento"].eq(codigo).sum()
        assert contagem_direta == linha[f"qtd_{codigo.lower()}_7d"]

display(amostras_tipos[[
    "chave_candidata", "data", "qtd_fo_7d", "qtd_lsd_7d",
    "qtd_alarmes_registrados_7d",
]])
print("Contagens FO e LSD conferidas com as ativações de origem.")


,chave_candidata,data,qtd_fo_7d,qtd_lsd_7d,qtd_alarmes_registrados_7d
0,BUT|1|10|PDM,2023-10-17,0,0,0
472031,VSO|2|9|PDM,2026-03-28,0,0,0
438550,VSO|1|1|PDM,2025-02-18,988,0,988
455432,VSO|2|15|PDM,2024-11-06,0,1936,1956


Contagens FO e LSD conferidas com as ativações de origem.


### Resultado dos históricos FO e LSD

A variável `tabela_mestra_tipos` acrescenta `qtd_fo_7d` e `qtd_lsd_7d`, preservando as contagens totais e o alvo parcial. O código recupera classificações ausentes a partir das TAGs e verifica sua consistência com os campos preenchidos.

As contagens continuam baseadas em ativações registradas, e os zeros não confirmam ausência de eventos na operação. Nenhum arquivo de dados é exportado e o treinamento permanece pendente dos rótulos negativos e da cobertura.

A próxima etapa deverá revisar os horários de normalização antes das variáveis de duração. Uma única ativação pode ter vários horários de normalização na fonte; selecionar um deles sem uma regra validada pode distorcer as anomalias de 15/30 segundos e as durações acumuladas.

## 42. Auditoria dos horários de normalização

Uma ativação continua sendo identificada por TAG + início. Esta seção conta quantos horários de normalização aparecem para cada ativação. As datas mínima e máxima são exibidas para auditoria, sem escolher arbitrariamente uma delas quando houver divergência.

A regra inicial admite o cálculo somente quando há **um fim distinto, nenhum fim ausente e nenhum fim anterior ao início**. Uma normalização posterior ao início da próxima ativação da mesma TAG também fica para revisão.

A classificação é feita sobre a extração disponível. Fim único significa consistência dentro desse arquivo, não comprovação da qualidade do registro nem de sua disponibilidade histórica.

In [45]:
registros_normalizacao = alarmes_identificados.loc[
    alarmes_identificados["chave_candidata"].isin(portas["chave_candidata"]),
    ["chave_candidata", "tag", "inicio_alarme", "fim_alarme"],
].copy()

registros_normalizacao["fim_ausente"] = registros_normalizacao["fim_alarme"].isna()
registros_normalizacao["fim_anterior_inicio"] = (
    registros_normalizacao["fim_alarme"] < registros_normalizacao["inicio_alarme"]
)

auditoria_duracoes = (
    registros_normalizacao.groupby(["chave_candidata", "tag", "inicio_alarme"])
    .agg(
        qtd_linhas=("fim_alarme", "size"),
        qtd_fins_distintos=("fim_alarme", "nunique"),
        primeiro_fim=("fim_alarme", "min"),
        ultimo_fim=("fim_alarme", "max"),
        tem_fim_ausente=("fim_ausente", "max"),
        tem_fim_anterior_inicio=("fim_anterior_inicio", "max"),
    )
    .reset_index()
    .sort_values(["tag", "inicio_alarme"])
    .reset_index(drop=True)
)

auditoria_duracoes["proximo_inicio"] = (
    auditoria_duracoes.groupby("tag")["inicio_alarme"].shift(-1)
)
auditoria_duracoes["fim_apos_proxima_ativacao"] = (
    auditoria_duracoes["primeiro_fim"] > auditoria_duracoes["proximo_inicio"]
)

auditoria_duracoes["fim_unico_utilizavel"] = (
    auditoria_duracoes["qtd_fins_distintos"].eq(1)
    & ~auditoria_duracoes["tem_fim_ausente"]
    & ~auditoria_duracoes["tem_fim_anterior_inicio"]
    & ~auditoria_duracoes["fim_apos_proxima_ativacao"]
)

assert len(auditoria_duracoes) == len(ativacoes_alarmes)
assert auditoria_duracoes["qtd_linhas"].sum() == len(registros_normalizacao)

duracoes_para_revisar = auditoria_duracoes.loc[
    ~auditoria_duracoes["fim_unico_utilizavel"]
].copy()

print("Ativações analisadas:", len(auditoria_duracoes))
print("Ativações com mais de um fim:", auditoria_duracoes["qtd_fins_distintos"].gt(1).sum())
print("Ativações com fim único utilizável:", auditoria_duracoes["fim_unico_utilizavel"].sum())
display(duracoes_para_revisar[[
    "tag", "inicio_alarme", "qtd_fins_distintos", "primeiro_fim", "ultimo_fim",
]].head())


Ativações analisadas: 304143
Ativações com mais de um fim: 54640
Ativações com fim único utilizável: 249503


,tag,inicio_alarme,qtd_fins_distintos,primeiro_fim,ultimo_fim
0,PSD_BUT_BUT1_PDM01_OBS,2024-01-21 23:12:46.257,33,2024-01-21 23:12:56.257,2024-02-20 18:02:26.770
2,PSD_BUT_BUT1_PDM01_OBS,2024-04-13 01:27:17.210,22,2024-04-18 01:14:28.663,2024-05-13 01:07:58.167
3,PSD_BUT_BUT1_PDM01_OBS,2024-05-23 18:15:43.987,12,2024-05-23 22:34:54.787,2024-06-03 13:29:14.897
5,PSD_BUT_BUT1_PDM01_OBS,2024-06-05 02:06:25.253,2,2024-06-05 04:53:13.300,2024-06-05 08:58:56.300
6,PSD_BUT_BUT1_PDM01_OBS,2024-06-08 11:16:31.223,2,2024-06-08 22:41:48.507,2024-06-09 01:02:46.830


## 43. Durações das ativações com fim único

A duração, em segundos, é calculada por fim − início, preservando as frações de segundo dos horários. A coluna de duração textual ou arredondada não é necessária para esse cálculo.

Nesta versão, uma anomalia de 15 segundos significa **duração maior ou igual a 15 segundos**. O mesmo critério é aplicado a 30 segundos. Esses limites são definições operacionais iniciais, não uma classificação automática de falha ou gravidade.

Durações elevadas permanecem no diagnóstico; sua magnitude não basta para classificá-las como erro. A tabela apresenta os percentis para orientar a revisão.

In [46]:
alarmes_fim_unico = auditoria_duracoes.loc[
    auditoria_duracoes["fim_unico_utilizavel"]
].copy()

# Com apenas um fim distinto, primeiro_fim e ultimo_fim são iguais.
assert alarmes_fim_unico["primeiro_fim"].equals(alarmes_fim_unico["ultimo_fim"])
alarmes_fim_unico["fim_alarme"] = alarmes_fim_unico["primeiro_fim"]
alarmes_fim_unico["duracao_seg"] = (
    alarmes_fim_unico["fim_alarme"] - alarmes_fim_unico["inicio_alarme"]
).dt.total_seconds()

alarmes_fim_unico["data_normalizacao"] = alarmes_fim_unico["fim_alarme"].dt.normalize()
alarmes_fim_unico["anomalia_15s"] = alarmes_fim_unico["duracao_seg"].ge(15).astype(int)
alarmes_fim_unico["anomalia_30s"] = alarmes_fim_unico["duracao_seg"].ge(30).astype(int)

assert alarmes_fim_unico["duracao_seg"].ge(0).all()
assert (alarmes_fim_unico["anomalia_30s"] <= alarmes_fim_unico["anomalia_15s"]).all()

display(alarmes_fim_unico["duracao_seg"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).to_frame("segundos"))


,segundos
count,2.495030e+05
mean,5.078661e+04
std,2.218203e+05
min,0.000000e+00
50%,8.000000e+00
90%,6.959157e+04
95%,2.751666e+05
99%,1.290313e+06
max,2.579599e+06


## 44. Agregação pelo dia da normalização

Um alarme iniciado em 10/01 e normalizado em 12/01 terá sua duração final contabilizada em **12/01**. A duração completa não poderia ser usada em uma previsão feita ao final de 10/01 ou 11/01.

As janelas desta etapa são, portanto, de **alarmes normalizados** em D−6…D ou D−13…D, e não de alarmes iniciados nesses períodos. Alarmes ainda abertos e casos ambíguos não entram nas durações calculadas.

A soma das durações pode contar intervalos sobrepostos e alarmes que não interrompem a operação. Por isso, ela recebe o nome `soma_duracoes_alarmes_7d` e **não substitui `tempo_parada_acumulado_7d`**.

Zero nas contagens e somas auxiliares significa nenhum caso elegível encontrado no arquivo. Máximos e médias sem casos permanecem ausentes.

In [47]:
duracoes_diarias = (
    alarmes_fim_unico.groupby(["chave_candidata", "data_normalizacao"])
    .agg(
        qtd_normalizados_fim_unico_1d=("duracao_seg", "size"),
        soma_duracoes_1d=("duracao_seg", "sum"),
        max_duracao_1d=("duracao_seg", "max"),
        qtd_anomalias_15s_1d=("anomalia_15s", "sum"),
        qtd_anomalias_30s_1d=("anomalia_30s", "sum"),
    )
    .reset_index()
    .rename(columns={"data_normalizacao": "data"})
)

historico_duracoes = historico_alarmes_diario[["chave_candidata", "data"]].merge(
    duracoes_diarias,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

contagens_duracoes = [
    "qtd_normalizados_fim_unico_1d", "qtd_anomalias_15s_1d", "qtd_anomalias_30s_1d",
]
historico_duracoes[contagens_duracoes] = (
    historico_duracoes[contagens_duracoes].fillna(0).astype(int)
)
historico_duracoes["soma_duracoes_1d"] = historico_duracoes["soma_duracoes_1d"].fillna(0)

historico_duracoes = (
    historico_duracoes.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
)

normalizados_no_historico = alarmes_fim_unico["data_normalizacao"].between(
    inicio_historico_alarmes, DATA_FIM
)
assert historico_duracoes["qtd_normalizados_fim_unico_1d"].sum() == normalizados_no_historico.sum()
display(historico_duracoes.loc[
    historico_duracoes["qtd_normalizados_fim_unico_1d"] > 0
].head())


,chave_candidata,data,qtd_normalizados_fim_unico_1d,soma_duracoes_1d,max_duracao_1d,qtd_anomalias_15s_1d,qtd_anomalias_30s_1d
37,BUT|1|10|PDM,2023-11-10,3,2464.983,2304.983,3,2
38,BUT|1|10|PDM,2023-11-11,3,24.970,14.984,0,0
39,BUT|1|10|PDM,2023-11-12,1,5.017,5.017,0,0
40,BUT|1|10|PDM,2023-11-13,10,640.004,470.000,4,2
48,BUT|1|10|PDM,2023-11-21,2,20.000,10.000,0,0


## 45. Janelas das durações e anomalias

As somas usam a mesma técnica `groupby().rolling()` das etapas anteriores. O máximo ignora dias sem casos elegíveis e permanece ausente quando toda a semana está sem casos.

A média é **soma das durações ÷ quantidade de alarmes normalizados elegíveis** na semana. Essa divisão evita dar o mesmo peso a um dia com um alarme e a outro com muitos alarmes.

Todas as variáveis desta seção usam apenas o subconjunto com fim único. A filtragem de qualidade foi feita na extração completa; o uso no modelo ainda exige confirmar quando os registros e suas revisões ficaram disponíveis.

In [48]:
# Cada linha define: coluna diária, tamanho da janela e nome do resultado.
calculos_duracao = [
    ("qtd_normalizados_fim_unico_1d", 7, "qtd_normalizados_fim_unico_7d"),
    ("qtd_normalizados_fim_unico_1d", 14, "qtd_normalizados_fim_unico_14d"),
    ("soma_duracoes_1d", 7, "soma_duracoes_alarmes_7d"),
    ("qtd_anomalias_15s_1d", 7, "qtd_anomalias_15s_7d"),
    ("qtd_anomalias_30s_1d", 14, "qtd_anomalias_30s_14d"),
]

for coluna_diaria, dias, coluna_resultado in calculos_duracao:
    historico_duracoes[coluna_resultado] = (
        historico_duracoes.groupby("chave_candidata")[coluna_diaria]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
    )

historico_duracoes["max_tempo_normalizacao_7d"] = (
    historico_duracoes.groupby("chave_candidata")["max_duracao_1d"]
    .rolling(window=7, min_periods=1)
    .max()
    .reset_index(level=0, drop=True)
)

quantidade_para_media = historico_duracoes["qtd_normalizados_fim_unico_7d"].where(
    historico_duracoes["qtd_normalizados_fim_unico_7d"] > 0
)
historico_duracoes["tempo_medio_por_alarme_7d"] = (
    historico_duracoes["soma_duracoes_alarmes_7d"] / quantidade_para_media
)

colunas_duracao = [
    "qtd_normalizados_fim_unico_7d", "qtd_normalizados_fim_unico_14d",
    "soma_duracoes_alarmes_7d", "qtd_anomalias_15s_7d", "qtd_anomalias_30s_14d",
    "max_tempo_normalizacao_7d", "tempo_medio_por_alarme_7d",
]

janelas_duracoes = historico_duracoes.loc[
    historico_duracoes["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_duracao],
].copy()

for coluna in [
    "qtd_normalizados_fim_unico_7d", "qtd_normalizados_fim_unico_14d",
    "qtd_anomalias_15s_7d", "qtd_anomalias_30s_14d",
]:
    janelas_duracoes[coluna] = janelas_duracoes[coluna].astype("Int64")

tabela_mestra_duracoes = tabela_mestra_tipos.merge(
    janelas_duracoes,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_duracoes) == len(tabela_mestra_tipos)
assert not tabela_mestra_duracoes.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_duracoes["falha_7d"].equals(tabela_mestra_tipos["falha_7d"])

print("Linhas da tabela:", len(tabela_mestra_duracoes))
display(tabela_mestra_duracoes[colunas_duracao].describe())


Linhas da tabela: 472032


,qtd_normalizados_fim_unico_7d,qtd_normalizados_fim_unico_14d,soma_duracoes_alarmes_7d,qtd_anomalias_15s_7d,qtd_anomalias_30s_14d,max_tempo_normalizacao_7d,tempo_medio_por_alarme_7d
count,472032.0,472032.0,4.720320e+05,472032.0,472032.0,1.382750e+05,1.382750e+05
mean,3.680212,7.352254,1.879074e+05,1.282947,1.847362,1.493221e+05,1.130610e+05
std,21.056035,36.299365,1.001635e+06,7.124325,7.290014,3.551900e+05,3.197673e+05
min,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
25%,0.0,0.0,0.000000e+00,0.0,0.0,5.016000e+00,5.000000e+00
50%,0.0,0.0,0.000000e+00,0.0,0.0,5.198400e+01,2.000000e+01
75%,2.0,5.0,9.972000e+00,0.0,0.0,1.189566e+05,4.214516e+04
max,1956.0,2507.0,1.805719e+07,723.0,502.0,2.579599e+06,2.579599e+06


## 46. Conferência por contagem e cálculo direto

A conferência usa o início e o fim do calendário e o dia com maior soma semanal. Para cada exemplo, o código seleciona diretamente os alarmes cuja normalização ocorreu no intervalo da janela.

O resultado das anomalias deve ser menor ou igual à quantidade de casos elegíveis na respectiva janela. Quando não há normalizações elegíveis na semana, média e máximo devem permanecer ausentes.

In [49]:
from math import isclose

assert (
    tabela_mestra_duracoes["qtd_anomalias_15s_7d"]
    <= tabela_mestra_duracoes["qtd_normalizados_fim_unico_7d"]
).all()
assert (
    tabela_mestra_duracoes["qtd_anomalias_30s_14d"]
    <= tabela_mestra_duracoes["qtd_normalizados_fim_unico_14d"]
).all()

sem_normalizacoes = tabela_mestra_duracoes["qtd_normalizados_fim_unico_7d"].eq(0)
assert tabela_mestra_duracoes.loc[
    sem_normalizacoes, ["max_tempo_normalizacao_7d", "tempo_medio_por_alarme_7d"]
].isna().all().all()

indices_duracao = [
    tabela_mestra_duracoes.index[0],
    tabela_mestra_duracoes.index[-1],
    tabela_mestra_duracoes["soma_duracoes_alarmes_7d"].idxmax(),
]
amostras_duracao = tabela_mestra_duracoes.loc[indices_duracao].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_duracao.iterrows():
    mesma_porta = alarmes_fim_unico["chave_candidata"].eq(linha["chave_candidata"])
    semana = alarmes_fim_unico.loc[
        mesma_porta & alarmes_fim_unico["data_normalizacao"].between(
            linha["data"] - pd.Timedelta(days=6), linha["data"]
        )
    ]
    quatorze_dias = alarmes_fim_unico.loc[
        mesma_porta & alarmes_fim_unico["data_normalizacao"].between(
            linha["data"] - pd.Timedelta(days=13), linha["data"]
        )
    ]

    assert len(semana) == linha["qtd_normalizados_fim_unico_7d"]
    assert len(quatorze_dias) == linha["qtd_normalizados_fim_unico_14d"]
    assert semana["duracao_seg"].ge(15).sum() == linha["qtd_anomalias_15s_7d"]
    assert quatorze_dias["duracao_seg"].ge(30).sum() == linha["qtd_anomalias_30s_14d"]
    assert isclose(semana["duracao_seg"].sum(), linha["soma_duracoes_alarmes_7d"], abs_tol=0.000001)

    # O fim de cada caso selecionado precede o início do dia seguinte à previsão.
    assert (semana["fim_alarme"] < linha["data"] + pd.Timedelta(days=1)).all()
    if len(semana) > 0:
        assert isclose(semana["duracao_seg"].mean(), linha["tempo_medio_por_alarme_7d"], abs_tol=0.000001)
        assert isclose(semana["duracao_seg"].max(), linha["max_tempo_normalizacao_7d"], abs_tol=0.000001)

display(amostras_duracao[["chave_candidata", "data", *colunas_duracao]])
print("Durações e anomalias conferidas com os casos de fim único.")


,chave_candidata,data,qtd_normalizados_fim_unico_7d,qtd_normalizados_fim_unico_14d,soma_duracoes_alarmes_7d,qtd_anomalias_15s_7d,qtd_anomalias_30s_14d,max_tempo_normalizacao_7d,tempo_medio_por_alarme_7d
0,BUT|1|10|PDM,2023-10-17,0,0,0.000000e+00,0,0,NaN,NaN
472031,VSO|2|9|PDM,2026-03-28,0,0,0.000000e+00,0,0,NaN,NaN
306823,PIN|1|17|PDM,2024-04-15,7,7,1.805719e+07,7,7,2579598.71,2.579599e+06


Durações e anomalias conferidas com os casos de fim único.


### Resultado das durações

A variável `tabela_mestra_duracoes` acrescenta as contagens de casos normalizados com fim único, as anomalias de 15/30 segundos, a soma, o máximo e a média de duração. As unidades de duração são segundos, e as janelas se referem ao **dia da normalização**.

Os casos ambíguos permanecem em `duracoes_para_revisar`. Durações elevadas continuam identificáveis em `alarmes_fim_unico`; sua validade precisa de revisão antes do treinamento. A soma de durações não equivale ao tempo de parada da porta.

O modelo ainda depende da cobertura e dos negativos do alvo. Também permanecem pendentes a confirmação da execução de ZM03, a classificação de alarmes graves e a interpretação operacional do tempo de parada. As variáveis produzidas até aqui são candidatas, sem treinamento ou exportação de dados.